In [ ]:
"""
SHD DEEP PROBE — pure-SNN MS-IF heterogeneity (vertical vs horizontal)
Experiment 1/3: smoke + speed + firing-stability check before DIAT headline.

Pure-SNN (neuromorphic-hardware target): inter-layer signal is spikes; no
BatchNorm/LayerNorm, no softmax gate. Per-type threshold calibration at init AND
per-epoch recalibration (gradient-free threshold homeostasis) hold firing at the
target throughout training; firing is logged every epoch as a drift check.

Paste into one Colab cell and Run. Resumes from checkpoint if interrupted.
"""

# ════════════════════════════════════════════════════════════════
# Imports + environment
# ════════════════════════════════════════════════════════════════
import os, sys, json, time, copy
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt   # Colab handles backend; never matplotlib.use('Agg')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ON_COLAB = True
except Exception:
    ON_COLAB = False

RESEARCH = Path('/content/drive/MyDrive/Research') if ON_COLAB else Path('.')
RESEARCH.mkdir(parents=True, exist_ok=True)
PROJECT = RESEARCH / 'NISAC_DeepHetero'
PROJECT.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# ════════════════════════════════════════════════════════════════
# Config (no argparse) + experiment manifest
# ════════════════════════════════════════════════════════════════
@dataclass
class Config:
    # architecture
    L: int = 4
    W: int = 128
    K: int = 5
    target_firing: float = 0.15
    stft_window_shd: int = 8          # SHD T=12 -> window must be <= T
    norm: str = "none"                # PURE SNN: no normalization layer
    # training
    n_epochs: int = 15
    lr: float = 1e-3
    batch_size: int = 128
    seeds: list = field(default_factory=lambda: [42, 123, 999])
    # firing-stability band (drift alarm)
    fire_lo: float = 0.02
    fire_hi: float = 0.90
    # methods to run on the probe
    methods: list = field(default_factory=lambda: [
        "D0_doppler", "D0_stft", "D0_chirp", "D0_dualtau", "D1_H1", "D1_rev", "D2_concat"])
    # experiment manifest (single source of truth)
    exp_index: int = 1
    exp_total: int = 3
    exp_name: str = "SHD deep probe"
    exp_manifest: list = field(default_factory=lambda: [
        (1, "SHD deep probe (T=12)", "THIS"),
        (2, "DIAT deep ablation (D0/D0+/D1/perm/D2)", "pending"),
        (3, "cross-family + gate variant", "pending"),
    ])

cfg = Config()

METHOD_LABELS = {
    "D0_doppler": "D0 homog Doppler-LIF",
    "D0_stft":    "D0 homog STFT-IF",
    "D0_chirp":   "D0 homog Chirp-LIF",
    "D0_dualtau": "D0 homog Dual-tau-LIF",
    "D1_H1":      "D1 vertical (STFT->Doppler->Chirp->Dualtau)",
    "D1_rev":     "D1 vertical reversed (falsification)",
    "D2_concat":  "D2 horizontal bank (concat)",
}
H1_ORDER = ["STFT-IF", "Doppler-LIF", "Chirp-LIF", "Dual-tau-LIF"]
H1_REV   = ["Dual-tau-LIF", "Chirp-LIF", "Doppler-LIF", "STFT-IF"]
D2_PATHS = ["Doppler-LIF", "Chirp-LIF", "STFT-IF", "Dual-tau-LIF", "CrossInhib-LIF"]


# ════════════════════════════════════════════════════════════════
# dikmen-spiking-neurons  (pip-install from GitHub if not present)
# ════════════════════════════════════════════════════════════════
def _clear_dikmen():
    for k in [k for k in sys.modules if k.startswith('dikmen')]:
        del sys.modules[k]

_clear_dikmen()
try:
    import dikmen_neurons  # noqa: F401  (already installed?)
except ImportError:
    import subprocess
    url = "git+https://github.com/DrCanD/dikmen-spiking-neurons.git"
    print("[SETUP] installing dikmen-spiking-neurons from GitHub ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", url])
    if r.returncode != 0:                      # PEP 668 externally-managed env
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--break-system-packages", url], check=True)
    _clear_dikmen()

from dikmen_neurons import NeuronRegistry   # 39 models / 8 families; we use MS-IF (ISAC)


# ════════════════════════════════════════════════════════════════
# Dataset registry + defensive SHD loader
# ════════════════════════════════════════════════════════════════
REGISTRY_PATH = RESEARCH / 'datasets.json'

def load_registry():
    if REGISTRY_PATH.exists():
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return {}

def register_dataset(name, path, **meta):
    reg = load_registry()
    reg[name] = {'path': str(path), 'registered': datetime.now().isoformat(), **meta}
    with open(REGISTRY_PATH, 'w') as f:
        json.dump(reg, f, indent=2)

def get_dataset_path(name):
    reg = load_registry()
    if name not in reg:
        raise KeyError(f"Dataset '{name}' not in registry {list(reg.keys())}. Register it.")
    p = Path(reg[name]['path'])
    if not p.exists():
        raise FileNotFoundError(f"'{name}' registered at {p} but not found.")
    return p

# register SHD once if missing (speaker-MIXED cache; not the official speaker-disjoint split)
if 'shd' not in load_registry():
    register_dataset('shd',
        path='/content/drive/MyDrive/Research/GHN_Instinct/gesture_cache/shd_T12_m200_v2.pt',
        source='zenkelab.org', format='pt-dict', classes=20, T=12, features=700,
        notes='Xtr/ytr/Xte/yte; speaker-mixed re-split, NOT official disjoint')

def load_shd():
    obj = torch.load(get_dataset_path('shd'), map_location='cpu', weights_only=False)
    if isinstance(obj, dict) and 'Xtr' in obj:
        Xtr, ytr, Xte, yte = obj['Xtr'], obj['ytr'], obj['Xte'], obj['yte']
    elif isinstance(obj, dict) and 'X' in obj:
        X, y = obj['X'], obj['y']
        n = len(X); idx = torch.randperm(n); cut = int(0.8 * n)
        Xtr, ytr = X[idx[:cut]], y[idx[:cut]]
        Xte, yte = X[idx[cut:]], y[idx[cut:]]
    else:
        raise ValueError(f"Unexpected SHD object keys: {getattr(obj,'keys',lambda:obj)()}")
    to_t = lambda a: torch.as_tensor(np.asarray(a)).float()
    to_y = lambda a: torch.as_tensor(np.asarray(a)).long()
    Xtr, Xte = to_t(Xtr), to_t(Xte)
    ytr, yte = to_y(ytr), to_y(yte)
    # SHD is [N, T=12, F=700] = [N, time, features]; no transpose needed
    assert Xtr.ndim == 3, f"expected [N,T,F], got {tuple(Xtr.shape)}"
    return Xtr, ytr, Xte, yte


# ════════════════════════════════════════════════════════════════
# Deep bodies (inline) — sandbox-verified
# ════════════════════════════════════════════════════════════════
def _make_norm(kind, width):
    if kind == "layer":
        return nn.LayerNorm(width)
    if kind == "batch":
        return nn.BatchNorm1d(width)
    return nn.Identity()           # PURE SNN default

class FeatureStack(nn.Module):
    """L (Linear -> [norm] -> neuron) layers; returns (time-avg feat [B,W], last spk [B,T,W])."""
    def __init__(self, in_features, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        neuron_kwargs = neuron_kwargs or {}
        dims = [in_features] + [width] * len(layer_types)
        self.projs = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(layer_types))])
        self.norms = nn.ModuleList([_make_norm(norm, width) for _ in layer_types])
        self.neurons = nn.ModuleList([NeuronRegistry.create(t, size=width, **neuron_kwargs.get(t, {}))
                                      for t in layer_types])
        self.width = width
    def forward(self, x):
        h = x
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z)
            h = spk
        return h.mean(dim=1), h
    def layer_firing(self, x):
        h, rates = x, []
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); rates.append(spk.float().mean().item()); h = spk
        return rates

class VerticalNet(nn.Module):
    def __init__(self, in_features, n_classes, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        self.stack = FeatureStack(in_features, layer_types, width, neuron_kwargs, norm)
        self.readout = nn.Linear(width, n_classes)
    def forward(self, x):
        feat, _ = self.stack(x); return self.readout(feat)
    def firing_rates(self, x):
        return self.stack.layer_firing(x)

class PathBankNet(nn.Module):
    def __init__(self, in_features, n_classes, path_types, width, n_layers,
                 fusion="concat", neuron_kwargs=None, norm="none", shared_stem=True):
        super().__init__()
        if shared_stem:
            self.stem = nn.Linear(in_features, width); path_in = width
        else:
            self.stem = None; path_in = in_features
        self.paths = nn.ModuleList([FeatureStack(path_in, [t]*n_layers, width, neuron_kwargs, norm)
                                    for t in path_types])
        self.readout = nn.Linear(width * len(path_types), n_classes)
        self.n_paths = len(path_types); self.width = width
    def forward(self, x):
        if self.stem is not None:
            B, T, d = x.shape; x = self.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        feats = [p(x)[0] for p in self.paths]
        return self.readout(torch.cat(feats, dim=-1))
    def path_firing(self, x):
        if self.stem is not None:
            B, T, d = x.shape; x = self.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        return [p(x)[1].float().mean().item() for p in self.paths]

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def _solve_width(builder, target, wmax=400):
    best = None
    for w in range(2, wmax):
        p = count_params(builder(w)); d = abs(p - target)
        if best is None or d < best[2]:
            best = (w, p, d)
    return best

@torch.no_grad()
def _calibrate_stack(stack, h, target, iters=30):
    for proj, norm, neuron in zip(stack.projs, stack.norms, stack.neurons):
        B, T, d = h.shape
        z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
        a, b = 1e-3, 1e5
        for _ in range(iters):
            mid = (a*b) ** 0.5; neuron.threshold = mid
            r = neuron(z)[0].float().mean().item()
            a, b = (mid, b) if r > target else (a, mid)
        neuron.threshold = (a*b) ** 0.5
        h = neuron(z)[0]
    return stack

@torch.no_grad()
def calibrate_thresholds(model, x, target):
    if isinstance(model, VerticalNet):
        _calibrate_stack(model.stack, x, target)
    else:
        h = x
        if model.stem is not None:
            B, T, d = x.shape; h = model.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        for p in model.paths:
            _calibrate_stack(p, h, target)
    return model


# ════════════════════════════════════════════════════════════════
# Model builder (capacity widths resolved at runtime against D1)
# ════════════════════════════════════════════════════════════════
def build_model(method, Fin, C, NK, matched):
    n = cfg.norm
    if method == "D0_doppler":
        return VerticalNet(Fin, C, ["Doppler-LIF"]*cfg.L, cfg.W, NK, n)
    if method == "D0_stft":
        return VerticalNet(Fin, C, ["STFT-IF"]*cfg.L, cfg.W, NK, n)
    if method == "D1_H1":
        return VerticalNet(Fin, C, H1_ORDER, cfg.W, NK, n)
    if method == "D1_rev":
        return VerticalNet(Fin, C, H1_REV, cfg.W, NK, n)
    if method == "D0_chirp":
        return VerticalNet(Fin, C, ["Chirp-LIF"]*cfg.L, cfg.W, NK, n)
    if method == "D0_dualtau":
        return VerticalNet(Fin, C, ["Dual-tau-LIF"]*cfg.L, cfg.W, NK, n)
    if method == "D2_concat":
        return PathBankNet(Fin, C, D2_PATHS, matched["d2_w"], cfg.L, "concat", NK, n, True)
    raise KeyError(method)


# ════════════════════════════════════════════════════════════════
# Train / eval / bootstrap
# ════════════════════════════════════════════════════════════════
def save_ckpt(model, opt, epoch, metrics, path):
    torch.save({'epoch': epoch, 'model_state_dict': copy.deepcopy(model.state_dict()),
                'optimizer_state_dict': opt.state_dict(), 'metrics': metrics}, path)

def load_ckpt(model, opt, path):
    if not path.exists():
        return 0, []
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state_dict']); opt.load_state_dict(ck['optimizer_state_dict'])
    print(f"    [RESUME] from epoch {ck['epoch']}")
    return ck['epoch'] + 1, ck['metrics'].get('fire_log', [])

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        correct.append((model(xb).argmax(1) == yb).int().cpu())
    return torch.cat(correct).numpy()

def firing_snapshot(model, xb):
    return model.firing_rates(xb) if isinstance(model, VerticalNet) else model.path_firing(xb)

def train_one(method, seed, Fin, C, NK, matched, tr_loader, te_loader, fire_batch, ckpt_dir):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = build_model(method, Fin, C, NK, matched).to(device)
    calibrate_thresholds(model, fire_batch.to(device), cfg.target_firing)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    ckpt = ckpt_dir / f'{method}_seed{seed}.pt'
    start, fire_log = load_ckpt(model, opt, ckpt)
    best_acc, best_corr = -1.0, None
    for ep in range(start, cfg.n_epochs):
        calibrate_thresholds(model, fire_batch.to(device), cfg.target_firing)  # per-epoch threshold homeostasis (gradient-free)
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward(); opt.step()
        corr = evaluate(model, te_loader); acc = float(corr.mean())
        fr = firing_snapshot(model, fire_batch.to(device))
        drift = not all(cfg.fire_lo < r < cfg.fire_hi for r in fr)
        fire_log.append({'epoch': ep, 'acc': acc, 'firing': [round(r, 4) for r in fr], 'drift': drift})
        if acc > best_acc:
            best_acc, best_corr = acc, corr
        save_ckpt(model, opt, ep, {'acc': acc, 'fire_log': fire_log}, ckpt)
        flag = "  !! FIRING DRIFT" if drift else ""
        print(f"    ep {ep:>2} acc={acc:.4f} firing={[round(r,3) for r in fr]}{flag}")
    return best_acc, best_corr, fire_log

def paired_bootstrap(corr_a, corr_b, nboot=5000, seed=0):
    rng = np.random.default_rng(seed)
    d = corr_a.astype(float) - corr_b.astype(float); n = len(d)
    boots = np.array([d[rng.integers(0, n, n)].mean() for _ in range(nboot)])
    return float(d.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


# ════════════════════════════════════════════════════════════════
# main
# ════════════════════════════════════════════════════════════════
def main():
    RUN_ID = datetime.now().strftime('%Y%m%d_%H%M')
    RUN_DIR = PROJECT / 'runs' / RUN_ID
    CKPT_DIR = RUN_DIR / 'checkpoints'
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

    # ── banner + manifest ──
    print(f"[PERSIST] project: {PROJECT}")
    print(f"[HW] device={device}  gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
    print(f"[PROGRESS] Experiment {cfg.exp_index}/{cfg.exp_total}: {cfg.exp_name}")
    for idx, name, status in cfg.exp_manifest:
        mark = "->" if status == "THIS" else ("x" if status == "done" else "  ")
        print(f"   {mark} {idx}/{cfg.exp_total}: {name} [{status}]")

    # ── data ──
    Xtr, ytr, Xte, yte = load_shd()
    Fin, C = Xtr.shape[-1], int(max(ytr.max(), yte.max()) + 1)
    print(f"[DATA] SHD train={tuple(Xtr.shape)} test={tuple(Xte.shape)} F={Fin} C={C}")
    print(f"[DATA] feature range [{Xtr.min():.3f}, {Xtr.max():.3f}]  (chance={1/C:.3f})")
    NK = {"STFT-IF": {"window_len": cfg.stft_window_shd}}
    fire_batch = Xtr[:cfg.batch_size]
    tr_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=cfg.batch_size, shuffle=True,
                           num_workers=2, pin_memory=(device.type == 'cuda'), persistent_workers=True)
    te_loader = DataLoader(TensorDataset(Xte, yte), batch_size=256, shuffle=False)

    # ── capacity matching against D1 (homogeneous D0_* at W=128 are already ~matched: neuron params are O(W)) ──
    d1_params = count_params(VerticalNet(Fin, C, H1_ORDER, cfg.W, NK, cfg.norm))
    d2_w, d2_p, _ = _solve_width(
        lambda w: PathBankNet(Fin, C, D2_PATHS, w, cfg.L, "concat", NK, cfg.norm, True), d1_params)
    matched = {"d2_w": d2_w}
    d0_dop_p = count_params(VerticalNet(Fin, C, ["Doppler-LIF"]*cfg.L, cfg.W, NK, cfg.norm))
    print(f"[CAPACITY] D1 params={d1_params}  (homog D0 at W={cfg.W} ~ {d0_dop_p}, {100*abs(d0_dop_p-d1_params)/d1_params:.2f}%)")
    print(f"[CAPACITY] D2 concat width={d2_w} -> {d2_p} ({100*abs(d2_p-d1_params)/d1_params:.2f}%)")

    # ── workload ──
    n_runs = len(cfg.methods) * len(cfg.seeds)
    print(f"[WORKLOAD] {len(cfg.methods)} methods x {len(cfg.seeds)} seeds = {n_runs} runs x {cfg.n_epochs} epochs")

    # ── run ──
    results = {}; run_counter = 0; first_t = None
    for method in cfg.methods:
        accs, corrs = [], []
        for seed in cfg.seeds:
            run_counter += 1
            print(f"\n  Run {run_counter}/{n_runs}: {METHOD_LABELS[method]} | seed={seed}")
            t0 = time.time()
            best_acc, best_corr, flog = train_one(method, seed, Fin, C, NK, matched,
                                                  tr_loader, te_loader, fire_batch, CKPT_DIR)
            dt = time.time() - t0
            if first_t is None:
                first_t = dt; print(f"    [ETA] ~{first_t*(n_runs-1)/60:.0f} min for remaining {n_runs-1} runs")
            accs.append(best_acc); corrs.append(best_corr)
            with open(RUN_DIR / f'results_{method}_seed{seed}.json', 'w') as f:
                json.dump({'method': method, 'seed': seed, 'best_acc': best_acc,
                           'corr': best_corr.tolist(), 'fire_log': flog, 'time_s': dt}, f, indent=2)
        results[method] = {'accs': accs, 'mean': float(np.mean(accs)), 'std': float(np.std(accs)),
                           'corrs': [c.tolist() for c in corrs]}
        print(f"  => {method}: acc {np.mean(accs):.4f} +/- {np.std(accs):.4f}")

    # ── paired bootstrap: control = capacity-matched BEST homogeneous (most stringent) ──
    homog = [m for m in results if m.startswith("D0_")]
    best_homog = max(homog, key=lambda m: results[m]['mean']) if homog else None
    if best_homog:
        print(f"\n[BOOTSTRAP] control = best homogeneous = {best_homog} ({results[best_homog]['mean']:.4f}); paired on test examples")
    else:
        print("\n[BOOTSTRAP] no homogeneous baseline present")

    def boot_pair(a, b):
        per_seed = [paired_bootstrap(np.array(results[a]['corrs'][si]),
                                     np.array(results[b]['corrs'][si]), seed=cfg.seeds[si])
                    for si in range(len(cfg.seeds))]
        diffs = [d for d, _, _ in per_seed]
        sig = all(lo > 0 for _, lo, _ in per_seed) or all(hi < 0 for _, _, hi in per_seed)
        print(f"   {a} - {b}: mean diff {np.mean(diffs):+.4f}  "
              f"per-seed CIs {[(round(l,3), round(h,3)) for _, l, h in per_seed]}  consistent_sign={sig}")
        return per_seed

    boot = {}
    if best_homog:
        for m in ["D1_H1", "D2_concat", "D1_rev"]:
            if m in results:
                boot[f"{m}_vs_bestHomog({best_homog})"] = boot_pair(m, best_homog)
    if "D2_concat" in results and "D1_H1" in results:
        boot["horizontal_vs_vertical"] = boot_pair("D2_concat", "D1_H1")     # the core question
    if "D1_H1" in results and "D1_rev" in results:
        boot["ordering_H1_vs_rev"] = boot_pair("D1_H1", "D1_rev")            # does layer order matter

    # ── summary ──
    summary = {'run_id': RUN_ID, 'exp_index': cfg.exp_index, 'config': {k: v for k, v in vars(cfg).items()},
               'capacity': {'d1_params': d1_params, 'd2_w': d2_w},
               'best_homog': best_homog,
               'results': {m: {'mean': r['mean'], 'std': r['std'], 'accs': r['accs']} for m, r in results.items()},
               'bootstrap': boot, 'timestamp': datetime.now().isoformat()}
    for name in ['summary.json', 'latest_summary.json']:
        with open((RUN_DIR if name == 'summary.json' else PROJECT) / name, 'w') as f:
            json.dump(summary, f, indent=2)

    # ── firing-stability plot (pure-SNN drift check) ──
    try:
        plt.figure(figsize=(7, 4))
        flog = json.load(open(RUN_DIR / f'results_D1_H1_seed{cfg.seeds[0]}.json'))['fire_log']
        arr = np.array([e['firing'] for e in flog])
        for li in range(arr.shape[1]):
            plt.plot(range(len(arr)), arr[:, li], marker='o', label=f'layer {li} ({H1_ORDER[li]})')
        plt.axhspan(cfg.fire_lo, cfg.fire_hi, color='green', alpha=0.06, label='healthy band')
        plt.xlabel('epoch'); plt.ylabel('firing rate'); plt.title('D1(H1) per-layer firing — pure-SNN stability')
        plt.legend(fontsize=7); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"[plot skipped] {e}")

    print(f"\n[DONE] summary -> {RUN_DIR / 'summary.json'}")
    return summary

summary = main()

In [ ]:
"""
DIAT-uSAT DEEP ABLATION — pure-SNN MS-IF heterogeneity (vertical vs horizontal)
Experiment 2/3: HEADLINE. Micro-Doppler radar (spectrogram-as-sequence, T=64).

Config A (accuracy-first): pure-SNN (inter-layer signal is spikes; no BatchNorm/
LayerNorm, no softmax gate). Per-type threshold calibration ONCE at init to avoid
dead/saturated layers; firing then runs free and is logged as an efficiency metric
(a sparsity-controlled variant is a separate secondary study). Capacity-matched,
paired bootstrap vs the best homogeneous baseline, pre-registered before results.

Paste into one Colab cell and Run. Resumes from checkpoint if interrupted.
"""

# ════════════════════════════════════════════════════════════════
# Imports + environment
# ════════════════════════════════════════════════════════════════
import os, sys, json, time, copy
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt   # Colab handles backend; never matplotlib.use('Agg')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ON_COLAB = True
except Exception:
    ON_COLAB = False

RESEARCH = Path('/content/drive/MyDrive/Research') if ON_COLAB else Path('.')
RESEARCH.mkdir(parents=True, exist_ok=True)
PROJECT = RESEARCH / 'NISAC_DeepHetero'
PROJECT.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# ════════════════════════════════════════════════════════════════
# Config (no argparse) + experiment manifest
# ════════════════════════════════════════════════════════════════
@dataclass
class Config:
    # architecture
    L: int = 4
    W: int = 128
    K: int = 5
    target_firing: float = 0.15
    stft_window: int = 16             # DIAT T=64 -> window <= T
    norm: str = "none"                # PURE SNN: no normalization layer
    # training (early stopping on validation: train to each config's own plateau)
    max_epochs: int = 200
    patience: int = 20                # stop after this many epochs with no smoothed-val improvement
    val_frac: float = 0.15            # validation carved from train (model selection ONLY)
    val_smooth: int = 5               # moving-average window on val score (denoise epoch selection)
    select_metric: str = "macro_f1"   # primary metric for early stopping + headline ("accuracy" | "macro_f1")
    lr: float = 1e-3
    batch_size: int = 128
    seeds: list = field(default_factory=lambda: [42, 123, 999])
    # firing band (logged as a diagnostic; NOT enforced during training under config A)
    fire_lo: float = 0.02
    fire_hi: float = 0.90
    # methods to run
    methods: list = field(default_factory=lambda: [
        "D0_doppler", "D0_stft", "D0_chirp", "D0_dualtau",
        "D1_H1", "D1_rev", "D1_perm1", "D2_concat"])
    # experiment manifest (single source of truth)
    exp_index: int = 2
    exp_total: int = 3
    exp_name: str = "DIAT-uSAT deep ablation (HEADLINE)"
    exp_manifest: list = field(default_factory=lambda: [
        (1, "SHD deep probe (T=12)", "done"),
        (2, "DIAT-uSAT deep ablation (D0/D1/perm/D2)", "THIS"),
        (3, "cross-family + sparsity-energy curve", "pending"),
    ])

cfg = Config()

METHOD_LABELS = {
    "D0_doppler": "D0 homog Doppler-LIF",
    "D0_stft":    "D0 homog STFT-IF",
    "D0_chirp":   "D0 homog Chirp-LIF",
    "D0_dualtau": "D0 homog Dual-tau-LIF",
    "D1_H1":      "D1 vertical (STFT->Doppler->Chirp->Dualtau)",
    "D1_rev":     "D1 vertical reversed (falsification)",
    "D1_perm1":   "D1 vertical permutation-1 (falsification)",
    "D2_concat":  "D2 horizontal bank (concat)",
}
H1_ORDER = ["STFT-IF", "Doppler-LIF", "Chirp-LIF", "Dual-tau-LIF"]
H1_REV   = ["Dual-tau-LIF", "Chirp-LIF", "Doppler-LIF", "STFT-IF"]
D1_PERM1 = ["Doppler-LIF", "Dual-tau-LIF", "STFT-IF", "Chirp-LIF"]
D2_PATHS = ["Doppler-LIF", "Chirp-LIF", "STFT-IF", "Dual-tau-LIF", "CrossInhib-LIF"]


# ════════════════════════════════════════════════════════════════
# dikmen-spiking-neurons  (pip-install from GitHub if not present)
# ════════════════════════════════════════════════════════════════
def _clear_dikmen():
    for k in [k for k in sys.modules if k.startswith('dikmen')]:
        del sys.modules[k]

_clear_dikmen()
try:
    import dikmen_neurons  # noqa: F401  (already installed?)
except ImportError:
    import subprocess
    url = "git+https://github.com/DrCanD/dikmen-spiking-neurons.git"
    print("[SETUP] installing dikmen-spiking-neurons from GitHub ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", url])
    if r.returncode != 0:                      # PEP 668 externally-managed env
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--break-system-packages", url], check=True)
    _clear_dikmen()

from dikmen_neurons import NeuronRegistry   # 39 models / 8 families; we use MS-IF (ISAC)


# ════════════════════════════════════════════════════════════════
# Dataset registry + DIAT folder loader
# ════════════════════════════════════════════════════════════════
REGISTRY_PATH = RESEARCH / 'datasets.json'

def load_registry():
    if REGISTRY_PATH.exists():
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return {}

def register_dataset(name, path, **meta):
    reg = load_registry()
    reg[name] = {'path': str(path), 'registered': datetime.now().isoformat(), **meta}
    with open(REGISTRY_PATH, 'w') as f:
        json.dump(reg, f, indent=2)

def get_dataset_path(name):
    reg = load_registry()
    if name not in reg:
        raise KeyError(f"Dataset '{name}' not in registry {list(reg.keys())}. Register it.")
    p = Path(reg[name]['path'])
    if not p.exists():
        raise FileNotFoundError(f"'{name}' registered at {p} but not found.")
    return p

# register DIAT-uSAT once if missing (folder with X.npy + y.npy)
if 'diat' not in load_registry():
    register_dataset('diat',
        path='/content/drive/MyDrive/NISAC/data/DIAT_uSAT/processed',
        source='DIAT micro-Doppler uSAT', format='folder-npy', classes=6, T=64, features=64,
        notes='X.npy [N,64,64] + y.npy; spectrogram-as-sequence; 80/20 stratified split seed=0')

def load_diat():
    base = get_dataset_path('diat')                       # folder containing X.npy + y.npy
    X = np.load(base / 'X.npy').astype(np.float32)         # [N, 64, 64] = [N, time, Doppler]
    y = np.load(base / 'y.npy').astype(np.int64)
    assert X.ndim == 3, f"expected [N,64,64], got {X.shape}"
    def strat_split(Xa, ya, frac, seed):
        try:
            from sklearn.model_selection import train_test_split
            A, B, ya2, yb2 = train_test_split(Xa, ya, test_size=frac, stratify=ya, random_state=seed)
            return A, ya2, B, yb2
        except Exception:
            rng = np.random.default_rng(seed); ia, ib = [], []
            for c in np.unique(ya):
                ci = np.where(ya == c)[0]; rng.shuffle(ci); k = int((1 - frac) * len(ci))
                ia += list(ci[:k]); ib += list(ci[k:])
            return Xa[ia], ya[ia], Xa[ib], ya[ib]
    Xtv, ytv, Xte, yte = strat_split(X, y, 0.20, 0)                    # 80% trainval / 20% test
    Xtr, ytr, Xva, yva = strat_split(Xtv, ytv, cfg.val_frac / 0.80, 0) # carve val from trainval
    mu, sd = float(Xtr.mean()), float(Xtr.std() + 1e-6)               # standardize with TRAIN stats only
    norm = lambda a: torch.as_tensor((a - mu) / sd).float()
    yy = lambda a: torch.as_tensor(a).long()
    return norm(Xtr), yy(ytr), norm(Xva), yy(yva), norm(Xte), yy(yte)


# ════════════════════════════════════════════════════════════════
# Deep bodies (inline) — sandbox-verified
# ════════════════════════════════════════════════════════════════
def _make_norm(kind, width):
    if kind == "layer":
        return nn.LayerNorm(width)
    if kind == "batch":
        return nn.BatchNorm1d(width)
    return nn.Identity()           # PURE SNN default

class FeatureStack(nn.Module):
    """L (Linear -> [norm] -> neuron) layers; returns (time-avg feat [B,W], last spk [B,T,W])."""
    def __init__(self, in_features, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        neuron_kwargs = neuron_kwargs or {}
        dims = [in_features] + [width] * len(layer_types)
        self.projs = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(layer_types))])
        self.norms = nn.ModuleList([_make_norm(norm, width) for _ in layer_types])
        self.neurons = nn.ModuleList([NeuronRegistry.create(t, size=width, **neuron_kwargs.get(t, {}))
                                      for t in layer_types])
        self.width = width
    def forward(self, x):
        h = x
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z)
            h = spk
        return h.mean(dim=1), h
    def layer_firing(self, x):
        h, rates = x, []
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); rates.append(spk.float().mean().item()); h = spk
        return rates

class VerticalNet(nn.Module):
    def __init__(self, in_features, n_classes, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        self.stack = FeatureStack(in_features, layer_types, width, neuron_kwargs, norm)
        self.readout = nn.Linear(width, n_classes)
    def forward(self, x):
        feat, _ = self.stack(x); return self.readout(feat)
    def firing_rates(self, x):
        return self.stack.layer_firing(x)

class PathBankNet(nn.Module):
    def __init__(self, in_features, n_classes, path_types, width, n_layers,
                 fusion="concat", neuron_kwargs=None, norm="none", shared_stem=True):
        super().__init__()
        if shared_stem:
            self.stem = nn.Linear(in_features, width); path_in = width
        else:
            self.stem = None; path_in = in_features
        self.paths = nn.ModuleList([FeatureStack(path_in, [t]*n_layers, width, neuron_kwargs, norm)
                                    for t in path_types])
        self.readout = nn.Linear(width * len(path_types), n_classes)
        self.n_paths = len(path_types); self.width = width
    def forward(self, x):
        if self.stem is not None:
            B, T, d = x.shape; x = self.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        feats = [p(x)[0] for p in self.paths]
        return self.readout(torch.cat(feats, dim=-1))
    def path_firing(self, x):
        if self.stem is not None:
            B, T, d = x.shape; x = self.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        return [p(x)[1].float().mean().item() for p in self.paths]

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def _solve_width(builder, target, wmax=400):
    best = None
    for w in range(2, wmax):
        p = count_params(builder(w)); d = abs(p - target)
        if best is None or d < best[2]:
            best = (w, p, d)
    return best

@torch.no_grad()
def _calibrate_stack(stack, h, target, iters=30):
    for proj, norm, neuron in zip(stack.projs, stack.norms, stack.neurons):
        B, T, d = h.shape
        z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
        a, b = 1e-3, 1e5
        for _ in range(iters):
            mid = (a*b) ** 0.5; neuron.threshold = mid
            r = neuron(z)[0].float().mean().item()
            a, b = (mid, b) if r > target else (a, mid)
        neuron.threshold = (a*b) ** 0.5
        h = neuron(z)[0]
    return stack

@torch.no_grad()
def calibrate_thresholds(model, x, target):
    if isinstance(model, VerticalNet):
        _calibrate_stack(model.stack, x, target)
    else:
        h = x
        if model.stem is not None:
            B, T, d = x.shape; h = model.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        for p in model.paths:
            _calibrate_stack(p, h, target)
    return model


# ════════════════════════════════════════════════════════════════
# Model builder (capacity widths resolved at runtime against D1)
# ════════════════════════════════════════════════════════════════
def build_model(method, Fin, C, NK, matched):
    n = cfg.norm
    if method == "D0_doppler":
        return VerticalNet(Fin, C, ["Doppler-LIF"]*cfg.L, cfg.W, NK, n)
    if method == "D0_stft":
        return VerticalNet(Fin, C, ["STFT-IF"]*cfg.L, cfg.W, NK, n)
    if method == "D1_H1":
        return VerticalNet(Fin, C, H1_ORDER, cfg.W, NK, n)
    if method == "D1_rev":
        return VerticalNet(Fin, C, H1_REV, cfg.W, NK, n)
    if method == "D1_perm1":
        return VerticalNet(Fin, C, D1_PERM1, cfg.W, NK, n)
    if method == "D0_chirp":
        return VerticalNet(Fin, C, ["Chirp-LIF"]*cfg.L, cfg.W, NK, n)
    if method == "D0_dualtau":
        return VerticalNet(Fin, C, ["Dual-tau-LIF"]*cfg.L, cfg.W, NK, n)
    if method == "D2_concat":
        return PathBankNet(Fin, C, D2_PATHS, matched["d2_w"], cfg.L, "concat", NK, n, True)
    raise KeyError(method)


# ════════════════════════════════════════════════════════════════
# Train / eval / bootstrap
# ════════════════════════════════════════════════════════════════
def save_ckpt(model, opt, epoch, metrics, path):
    torch.save({'epoch': epoch, 'model_state_dict': copy.deepcopy(model.state_dict()),
                'optimizer_state_dict': opt.state_dict(), 'metrics': metrics}, path)

def load_ckpt(model, opt, path):
    if not path.exists():
        return 0, {}
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state_dict']); opt.load_state_dict(ck['optimizer_state_dict'])
    print(f"    [RESUME] from epoch {ck['epoch']}")
    return ck['epoch'] + 1, ck['metrics']

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); preds, trues = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(device)).argmax(1).cpu()); trues.append(yb)
    return torch.cat(preds).numpy(), torch.cat(trues).numpy()

def accuracy(preds, trues, C=None):
    return float((preds == trues).mean())

def macro_f1(preds, trues, C):
    f1s = []
    for c in range(C):
        tp = int(np.sum((preds == c) & (trues == c)))
        fp = int(np.sum((preds == c) & (trues != c)))
        fn = int(np.sum((preds != c) & (trues == c)))
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1s.append(2 * p * r / (p + r) if (p + r) else 0.0)
    return float(np.mean(f1s)), [round(float(x), 4) for x in f1s]

METRICS = {"accuracy": lambda p, t, C: accuracy(p, t),
           "macro_f1": lambda p, t, C: macro_f1(p, t, C)[0]}

def firing_snapshot(model, xb):
    return model.firing_rates(xb) if isinstance(model, VerticalNet) else model.path_firing(xb)

def train_one(method, seed, Fin, C, NK, matched, tr_loader, va_loader, te_loader, fire_batch, ckpt_dir):
    """Early stopping on SMOOTHED validation (moving avg, denoised); report TEST once at the
    best-smoothed-val epoch (selection on val only -> no test peeking)."""
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = build_model(method, Fin, C, NK, matched).to(device)
    calibrate_thresholds(model, fire_batch.to(device), cfg.target_firing)   # init only (config A)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    ckpt = ckpt_dir / f'{method}_seed{seed}.pt'
    start, m = load_ckpt(model, opt, ckpt)
    sel = METRICS[cfg.select_metric]
    fire_log     = m.get('fire_log', [])
    val_hist     = m.get('val_hist', [])     # raw per-epoch val scores (for the moving average)
    best_val     = m.get('best_val', -1.0)   # best SMOOTHED val
    best_epoch   = m.get('best_epoch', -1)
    patience_ctr = m.get('patience_ctr', 0)
    best = m.get('best', None)   # {'acc','f1','f1_per_class','preds','trues'} at best-smoothed-val epoch
    for ep in range(start, cfg.max_epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward(); opt.step()
        vp, vt = evaluate(model, va_loader); v_raw = sel(vp, vt, C)
        val_hist.append(v_raw)
        v_smooth = float(np.mean(val_hist[-cfg.val_smooth:]))   # denoised selection signal
        tp, tt = evaluate(model, te_loader)
        t_acc = accuracy(tp, tt); t_f1, t_f1c = macro_f1(tp, tt, C)
        fr = firing_snapshot(model, fire_batch.to(device))
        improved = v_smooth > best_val + 1e-4
        if improved:                                            # selection on smoothed validation only
            best_val, best_epoch, patience_ctr = v_smooth, ep, 0
            best = {'acc': t_acc, 'f1': t_f1, 'f1_per_class': t_f1c,
                    'preds': tp.tolist(), 'trues': tt.tolist()}  # test captured AT best-smoothed-val epoch
        else:
            patience_ctr += 1
        fire_log.append({'epoch': ep, f'val_{cfg.select_metric}_smooth': round(v_smooth, 4),
                         f'val_{cfg.select_metric}_raw': round(v_raw, 4),
                         'test_acc': round(t_acc, 4), 'test_f1': round(t_f1, 4),
                         'firing': [round(r, 4) for r in fr]})
        save_ckpt(model, opt, ep, {'fire_log': fire_log, 'val_hist': val_hist, 'best_val': best_val,
                                   'best_epoch': best_epoch, 'patience_ctr': patience_ctr,
                                   'best': best}, ckpt)
        star = " *" if improved else ""
        print(f"    ep {ep:>3} val={v_smooth:.4f}(raw {v_raw:.4f}) test_acc={t_acc:.4f} test_f1={t_f1:.4f}{star}")
        if patience_ctr >= cfg.patience:
            print(f"    [EARLY STOP] no smoothed-val gain {cfg.patience} ep; best val={best_val:.4f} @ep{best_epoch} "
                  f"-> test acc={best['acc']:.4f} f1={best['f1']:.4f}")
            break
    return best, fire_log

def paired_bootstrap(preds_a, preds_b, trues, metric_fn, C, nboot=2000, seed=0):
    """Paired bootstrap over test examples; recomputes the metric on each resample."""
    rng = np.random.default_rng(seed); n = len(trues)
    base = metric_fn(preds_a, trues, C) - metric_fn(preds_b, trues, C)
    boots = np.empty(nboot)
    for k in range(nboot):
        i = rng.integers(0, n, n)
        boots[k] = metric_fn(preds_a[i], trues[i], C) - metric_fn(preds_b[i], trues[i], C)
    return float(base), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


# ════════════════════════════════════════════════════════════════
# main
# ════════════════════════════════════════════════════════════════
def main():
    RUN_ID = datetime.now().strftime('%Y%m%d_%H%M')
    RUN_DIR = PROJECT / 'runs' / RUN_ID
    CKPT_DIR = RUN_DIR / 'checkpoints'
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

    # ── banner + manifest ──
    print(f"[PERSIST] project: {PROJECT}")
    print(f"[HW] device={device}  gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
    print(f"[PROGRESS] Experiment {cfg.exp_index}/{cfg.exp_total}: {cfg.exp_name}")
    for idx, name, status in cfg.exp_manifest:
        mark = "->" if status == "THIS" else ("x" if status == "done" else "  ")
        print(f"   {mark} {idx}/{cfg.exp_total}: {name} [{status}]")

    # ── data (train / val / test; val is for early-stopping model selection only) ──
    Xtr, ytr, Xva, yva, Xte, yte = load_diat()
    Fin, C = Xtr.shape[-1], int(max(ytr.max(), yva.max(), yte.max()) + 1)
    print(f"[DATA] DIAT train={tuple(Xtr.shape)} val={tuple(Xva.shape)} test={tuple(Xte.shape)} F={Fin} C={C}")
    print(f"[DATA] standardized range [{Xtr.min():.3f}, {Xtr.max():.3f}]  (chance={1/C:.3f})")
    NK = {"STFT-IF": {"window_len": cfg.stft_window}}
    fire_batch = Xtr[:cfg.batch_size]
    tr_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=cfg.batch_size, shuffle=True,
                           num_workers=2, pin_memory=(device.type == 'cuda'), persistent_workers=True)
    va_loader = DataLoader(TensorDataset(Xva, yva), batch_size=256, shuffle=False)
    te_loader = DataLoader(TensorDataset(Xte, yte), batch_size=256, shuffle=False)

    # ── capacity matching against D1 (homogeneous D0_* at W=128 are already ~matched: neuron params are O(W)) ──
    d1_params = count_params(VerticalNet(Fin, C, H1_ORDER, cfg.W, NK, cfg.norm))
    d2_w, d2_p, _ = _solve_width(
        lambda w: PathBankNet(Fin, C, D2_PATHS, w, cfg.L, "concat", NK, cfg.norm, True), d1_params)
    matched = {"d2_w": d2_w}
    d0_dop_p = count_params(VerticalNet(Fin, C, ["Doppler-LIF"]*cfg.L, cfg.W, NK, cfg.norm))
    print(f"[CAPACITY] D1 params={d1_params}  (homog D0 at W={cfg.W} ~ {d0_dop_p}, {100*abs(d0_dop_p-d1_params)/d1_params:.2f}%)")
    print(f"[CAPACITY] D2 concat width={d2_w} -> {d2_p} ({100*abs(d2_p-d1_params)/d1_params:.2f}%)")

    # ── PRE-REGISTERED predictions (printed before any result is seen) ──
    print("[PRE-REGISTERED] directional bets, fixed before results:")
    print("   P1: D2 (horizontal) > best homogeneous, sign consistent across seeds")
    print("   P2: D2 > D1 (horizontal beats vertical)")
    print("   P3: D1(H1) > reverse and permutation (layer order matters)")
    print("   P4: D1 vs best-homog is LESS negative than on SHD (STFT is in-domain on radar)")
    print("   P5: best homogeneous is a spectro-temporal resonator (Doppler/Chirp/STFT), not worst")

    # ── workload ──
    n_runs = len(cfg.methods) * len(cfg.seeds)
    print(f"[WORKLOAD] {len(cfg.methods)} methods x {len(cfg.seeds)} seeds = {n_runs} runs; "
          f"early stop on val (patience {cfg.patience}, max {cfg.max_epochs} ep)")

    # ── run ──
    results = {}; run_counter = 0; first_t = None
    for method in cfg.methods:
        accs, f1s, preds, trues, f1cs = [], [], [], [], []
        for seed in cfg.seeds:
            run_counter += 1
            print(f"\n  Run {run_counter}/{n_runs}: {METHOD_LABELS[method]} | seed={seed}")
            t0 = time.time()
            best, flog = train_one(method, seed, Fin, C, NK, matched,
                                   tr_loader, va_loader, te_loader, fire_batch, CKPT_DIR)
            dt = time.time() - t0
            if first_t is None:
                first_t = dt; print(f"    [ETA] ~{first_t*(n_runs-1)/60:.0f} min for remaining {n_runs-1} runs (early stop varies)")
            accs.append(best['acc']); f1s.append(best['f1']); f1cs.append(best['f1_per_class'])
            preds.append(best['preds']); trues.append(best['trues'])
            with open(RUN_DIR / f'results_{method}_seed{seed}.json', 'w') as f:
                json.dump({'method': method, 'seed': seed, 'acc': best['acc'], 'f1': best['f1'],
                           'f1_per_class': best['f1_per_class'], 'preds': best['preds'],
                           'trues': best['trues'], 'fire_log': flog, 'time_s': dt}, f, indent=2)
        results[method] = {'accs': accs, 'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)),
                           'f1s': f1s, 'f1_mean': float(np.mean(f1s)), 'f1_std': float(np.std(f1s)),
                           'f1_per_class': f1cs, 'preds': preds, 'trues': trues}
        print(f"  => {method}: acc {np.mean(accs):.4f}+/-{np.std(accs):.4f}  "
              f"macroF1 {np.mean(f1s):.4f}+/-{np.std(f1s):.4f}  per-class-F1[seed0]={f1cs[0]}")

    # ── paired bootstrap: control = capacity-matched BEST homogeneous by PRIMARY metric (most stringent) ──
    primary = 'f1_mean' if cfg.select_metric == 'macro_f1' else 'acc_mean'
    homog = [m for m in results if m.startswith("D0_")]
    best_homog = max(homog, key=lambda m: results[m][primary]) if homog else None
    if best_homog:
        print(f"\n[BOOTSTRAP] control = best homogeneous = {best_homog} "
              f"(acc {results[best_homog]['acc_mean']:.4f}, F1 {results[best_homog]['f1_mean']:.4f}); paired on test, both metrics")
    else:
        print("\n[BOOTSTRAP] no homogeneous baseline present")

    def boot_pair(a, b):
        out = {}
        for label, mfn in [("F1", METRICS["macro_f1"]), ("acc", METRICS["accuracy"])]:
            per_seed = [paired_bootstrap(np.array(results[a]['preds'][si]), np.array(results[b]['preds'][si]),
                                         np.array(results[a]['trues'][si]), mfn, C, seed=cfg.seeds[si])
                        for si in range(len(cfg.seeds))]
            diffs = [d for d, _, _ in per_seed]
            sig = all(lo > 0 for _, lo, _ in per_seed) or all(hi < 0 for _, _, hi in per_seed)
            print(f"   {a} - {b} [{label}]: mean {np.mean(diffs):+.4f}  "
                  f"CIs {[(round(l,3), round(h,3)) for _, l, h in per_seed]}  consistent={sig}")
            out[label] = per_seed
        return out

    boot = {}
    if best_homog:
        for m in ["D1_H1", "D2_concat", "D1_rev"]:
            if m in results:
                boot[f"{m}_vs_bestHomog({best_homog})"] = boot_pair(m, best_homog)
    if "D2_concat" in results and "D1_H1" in results:
        boot["horizontal_vs_vertical"] = boot_pair("D2_concat", "D1_H1")     # the core question
    if "D1_H1" in results and "D1_rev" in results:
        boot["ordering_H1_vs_rev"] = boot_pair("D1_H1", "D1_rev")            # does layer order matter
    if "D1_H1" in results and "D1_perm1" in results:
        boot["ordering_H1_vs_perm1"] = boot_pair("D1_H1", "D1_perm1")

    # ── summary ──
    summary = {'run_id': RUN_ID, 'exp_index': cfg.exp_index, 'select_metric': cfg.select_metric,
               'config': {k: v for k, v in vars(cfg).items()},
               'capacity': {'d1_params': d1_params, 'd2_w': d2_w}, 'best_homog': best_homog,
               'results': {m: {'acc_mean': r['acc_mean'], 'acc_std': r['acc_std'], 'accs': r['accs'],
                               'f1_mean': r['f1_mean'], 'f1_std': r['f1_std'], 'f1s': r['f1s'],
                               'f1_per_class': r['f1_per_class']} for m, r in results.items()},
               'bootstrap': boot, 'timestamp': datetime.now().isoformat()}
    for name in ['summary.json', 'latest_summary.json']:
        with open((RUN_DIR if name == 'summary.json' else PROJECT) / name, 'w') as f:
            json.dump(summary, f, indent=2)

    # ── firing-stability plot (pure-SNN drift check) ──
    try:
        plt.figure(figsize=(7, 4))
        flog = json.load(open(RUN_DIR / f'results_D1_H1_seed{cfg.seeds[0]}.json'))['fire_log']
        arr = np.array([e['firing'] for e in flog])
        for li in range(arr.shape[1]):
            plt.plot(range(len(arr)), arr[:, li], marker='o', label=f'layer {li} ({H1_ORDER[li]})')
        plt.axhspan(cfg.fire_lo, cfg.fire_hi, color='green', alpha=0.06, label='healthy band')
        plt.xlabel('epoch'); plt.ylabel('firing rate'); plt.title('D1(H1) per-layer firing (config A: init-calibrated, free during training)')
        plt.legend(fontsize=7); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"[plot skipped] {e}")

    print(f"\n[DONE] summary -> {RUN_DIR / 'summary.json'}")
    return summary

summary = main()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DIAT-uSAT  ·  BEST-MODEL SEARCH  (application phase, separate from the ablation)
# Multi-objective HPO:  maximize test macro-F1   ·   minimize test mean-firing
# Candidates:  Doppler-homog   ·   D2 path-bank (concat)   ·   Vanilla-LIF (BASELINE)
#
# Design (locked):
#   - Baseline = matched Vanilla-LIF: same BaseNeuron interface, same FastSigmoid
#     surrogate, plain leaky-integrate-fire dynamics, ZERO learnable neuron params.
#     Isolates the spectro-temporal inductive bias, not the implementation.
#   - Selection machinery is IDENTICAL to the ablation: per-epoch threshold
#     calibration at init (config A), early-stop on SMOOTHED validation macro-F1,
#     test measured ONCE at the best-smoothed-val epoch (no test peeking).
#   - HPO runs on a single seed (fast); the Pareto-selected configs are then
#     RE-EVALUATED on 3 seeds for the final report + paired bootstrap.
#   - Pruning is a conservative ABSOLUTE floor (won't kill late-convergers:
#     in the ablation, strong configs only converged at ep 130-190).
#   - param-count is logged per trial as a third column (FPGA/area budget).
#
# Single paste-and-run Colab cell. SQLite storage → resumes across sessions.
# ══════════════════════════════════════════════════════════════════════════════
import os, json, math, copy, time, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

# ── Colab vs sandbox ──────────────────────────────────────────────────────────
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ────────────────────────────── CONFIG (overridable) ──────────────────────────
class HPO:
    # search space (8 high-impact dims; wd/batch held at sane defaults for the main search)
    W_CHOICES        = [96, 128, 192, 256, 384]     # width — deployable cap, not 2048
    L_CHOICES        = [3, 4, 5, 6]                  # depth
    BETA_CHOICES     = [0.85, 0.90, 0.95, 0.97, 0.99]  # membrane decay (integration window)
    LR_LOW, LR_HIGH  = 3e-4, 4e-3                    # log-uniform
    SCHED_CHOICES    = ["none", "cosine"]           # none | warmup+cosine
    SURR_CHOICES     = [10.0, 25.0, 50.0]           # FastSigmoid surrogate slope (global attr)
    FIRE_CHOICES     = [0.10, 0.15, 0.20]           # calibration target -> threshold; also moves cost axis
    INPUT_CHOICES    = ["plain", "log1p"]           # input transform (radar power dynamic range)
    WEIGHT_DECAY     = 0.0                           # fixed for main search
    BATCH            = 128                           # fixed for main search

    # objectives / training
    MAX_EPOCHS       = 200
    PATIENCE         = 20
    VAL_SMOOTH       = 5
    VAL_FRAC         = 0.15
    SELECT_METRIC    = "macro_f1"                    # consistent with ablation

    # budget
    N_TRIALS         = 55                            # per architecture
    HPO_SEED         = 42                            # single seed for the search
    REEVAL_SEEDS     = [42, 123, 999]                # multi-seed re-eval of selected configs
    ARCHS            = ["doppler", "d2", "lif"]      # lif = baseline

    # visibility (so the run is not a black box)
    USE_TQDM         = True                          # live per-epoch progress bar inside each trial
    HEARTBEAT_EVERY  = 5                             # if no tqdm: print a status line every N epochs

    # conservative absolute-floor pruning (safe vs late convergence)
    PRUNE_EPOCH      = 30
    PRUNE_FLOOR      = 0.30                          # 6-class chance=0.167; collapse (STFT) stays ~0.15-0.18

    # paths
    if IN_COLAB:
        PROJECT  = "/content/drive/MyDrive/Research/NISAC_DeepHetero"
        DATA_DIR = "/content/drive/MyDrive/NISAC/data/DIAT_uSAT/processed"
    else:
        PROJECT  = "/home/claude/hpo_run"
        DATA_DIR = "/home/claude/syn_diat"

cfg = HPO()

# ── ACTIVE RUN SCOPE ──────────────────────────────────────────────────────────
# D2 is already dominated: its 8 trials show the whole D2 front sits at LOWER F1
# and HIGHER firing than Doppler's, even though D2 is NOT capacity-matched here
# (5x path params at a given W). Finishing its 55 (~2-3 h/trial on T4) buys nothing.
# Doppler is done (55/55 -> resumes at 0-todo). The crux left is LIF (the baseline),
# which is fast (homogeneous, Doppler-speed). TPE locked the good region by ~trial
# 17, so 30 trials is plenty. Set RUN_ID unchanged so everything resumes in place.
cfg.ARCHS    = ["doppler", "lif"]
cfg.N_TRIALS = 30

# ── reduced sandbox overrides (set REDUCED=1 in env for a fast end-to-end check) ─
if os.environ.get("REDUCED") == "1":
    cfg.W_CHOICES   = [16, 24, 32]
    cfg.L_CHOICES   = [2, 3]
    cfg.MAX_EPOCHS  = 6
    cfg.PATIENCE    = 4
    cfg.VAL_SMOOTH  = 2
    cfg.N_TRIALS    = 3
    cfg.REEVAL_SEEDS = [42]
    cfg.PRUNE_EPOCH = 99          # don't prune in the tiny check
    cfg.ARCHS       = ["doppler", "d2", "lif"]

# ── Colab-only setup ──────────────────────────────────────────────────────────
if IN_COLAB:
    os.system("pip install -q git+https://github.com/DrCanD/dikmen-spiking-neurons.git optuna")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import optuna
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False
import dikmen_neurons as D
from dikmen_neurons import BaseNeuron, spike_hard, NeuronRegistry

# FastSigmoidSurrogate lives in the submodule dikmen_neurons.base (NOT top-level).
# spike_hard() calls it from its own module globals, so grab the exact class object
# there; setting .scale on it is what backward() reads (class-attribute lookup).
def _locate_surrogate():
    c = getattr(D, "FastSigmoidSurrogate", None)
    if c is not None:
        return c
    sh = getattr(D, "spike_hard", None)
    if sh is not None and hasattr(sh, "__globals__"):
        c = sh.__globals__.get("FastSigmoidSurrogate")
        if c is not None:
            return c
    try:
        import pkgutil, importlib
        for _, name, _ in pkgutil.walk_packages(D.__path__, D.__name__ + "."):
            m = importlib.import_module(name)
            if hasattr(m, "FastSigmoidSurrogate"):
                return getattr(m, "FastSigmoidSurrogate")
    except Exception:
        pass
    return None

SURROGATE = _locate_surrogate()
if SURROGATE is None:
    print("[WARN] FastSigmoidSurrogate not found; surrogate-slope tuning disabled (package default).")

device = "cuda" if torch.cuda.is_available() else "cpu"
Path(cfg.PROJECT).mkdir(parents=True, exist_ok=True)
RUN_ID  = os.environ.get("RUN_ID", "hpo_main")   # STABLE across re-runs -> resume; change for a new experiment
RUN_DIR = Path(cfg.PROJECT) / "hpo" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = RUN_DIR / "ckpt"; CKPT_DIR.mkdir(exist_ok=True)
STORAGE  = f"sqlite:///{RUN_DIR / 'studies.db'}"
print(f"[HW] device={device}" + (f"  gpu={torch.cuda.get_device_name(0)}" if device == 'cuda' else ""))
print(f"[RUN] {RUN_DIR}", flush=True)

# ══════════════════════════════════════════════════════════════════════════════
# Matched Vanilla-LIF baseline  (registered into the package registry)
# ══════════════════════════════════════════════════════════════════════════════
class VanillaLIF(BaseNeuron):
    _family = "lif"
    _description = "Standard leaky integrate-and-fire baseline (no resonance/freq-selectivity)."
    def single_step(self, x_t, state):
        mem = self.beta * state["mem"] + x_t
        spk = spike_hard(mem, self.threshold)
        mem = mem * (1.0 - spk)
        return spk, {"mem": mem}
NeuronRegistry._all["Vanilla-LIF"] = VanillaLIF

D2_PATHS = ["Doppler-LIF", "Chirp-LIF", "STFT-IF", "Dual-tau-LIF", "CrossInhib-LIF"]

# ══════════════════════════════════════════════════════════════════════════════
# Deep bodies (inline) — identical to the ablation harness
# ══════════════════════════════════════════════════════════════════════════════
def _make_norm(kind, width):
    if kind == "layer": return nn.LayerNorm(width)
    if kind == "batch": return nn.BatchNorm1d(width)
    return nn.Identity()                      # PURE SNN default

class FeatureStack(nn.Module):
    def __init__(self, in_features, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        neuron_kwargs = neuron_kwargs or {}
        dims = [in_features] + [width] * len(layer_types)
        self.projs = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(layer_types))])
        self.norms = nn.ModuleList([_make_norm(norm, width) for _ in layer_types])
        self.neurons = nn.ModuleList(
            [NeuronRegistry.create(t, size=width, **neuron_kwargs.get(t, {})) for t in layer_types])
        self.width = width
    def forward(self, x):
        h = x
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); h = spk
        return h.mean(dim=1), h
    def layer_firing(self, x):
        h, rates = x, []
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); rates.append(spk.float().mean().item()); h = spk
        return rates

class VerticalNet(nn.Module):
    def __init__(self, in_features, n_classes, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        self.stack = FeatureStack(in_features, layer_types, width, neuron_kwargs, norm)
        self.readout = nn.Linear(width, n_classes)
    def forward(self, x):
        feat, _ = self.stack(x); return self.readout(feat)
    def firing_rates(self, x): return self.stack.layer_firing(x)

class PathBankNet(nn.Module):
    def __init__(self, in_features, n_classes, path_types, width, n_layers,
                 fusion="concat", neuron_kwargs=None, norm="none", shared_stem=True):
        super().__init__()
        if shared_stem:
            self.stem = nn.Linear(in_features, width); path_in = width
        else:
            self.stem = None; path_in = in_features
        self.paths = nn.ModuleList(
            [FeatureStack(path_in, [t]*n_layers, width, neuron_kwargs, norm) for t in path_types])
        self.fusion = fusion; self.width = width
        feat_dim = width * len(path_types)
        self.readout = nn.Linear(feat_dim, n_classes)
    def forward(self, x):
        if self.stem is not None:
            B, T, d = x.shape; x = self.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        feats = [p(x)[0] for p in self.paths]
        return self.readout(torch.cat(feats, dim=-1))
    def path_firing(self, x):
        if self.stem is not None:
            B, T, d = x.shape; x = self.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        return [p(x)[1].float().mean().item() for p in self.paths]

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def _calibrate_stack(stack, h, target, iters=30):
    for proj, norm, neuron in zip(stack.projs, stack.norms, stack.neurons):
        B, T, d = h.shape
        z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
        a, b = 1e-3, 1e5
        for _ in range(iters):
            mid = (a*b)**0.5; neuron.threshold = mid
            r = neuron(z)[0].float().mean().item()
            if r > target: a = mid
            else:          b = mid
        neuron.threshold = (a*b)**0.5
        h = neuron(z)[0]
    return stack

@torch.no_grad()
def calibrate_thresholds(model, x, target):
    if isinstance(model, VerticalNet):
        _calibrate_stack(model.stack, x, target)
    else:
        h = x
        if model.stem is not None:
            B, T, d = x.shape; h = model.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        for p in model.paths: _calibrate_stack(p, h, target)
    return model

# ══════════════════════════════════════════════════════════════════════════════
# Data  (precompute plain + log1p standardized splits once; train-stat standardize)
# ══════════════════════════════════════════════════════════════════════════════
def _strat_split(Xa, ya, frac, seed):
    try:
        from sklearn.model_selection import train_test_split
        A, B, ya2, yb2 = train_test_split(Xa, ya, test_size=frac, stratify=ya, random_state=seed)
        return A, ya2, B, yb2
    except Exception:
        rng = np.random.default_rng(seed); ia, ib = [], []
        for c in np.unique(ya):
            ci = np.where(ya == c)[0]; rng.shuffle(ci); k = int((1-frac)*len(ci))
            ia += list(ci[:k]); ib += list(ci[k:])
        return Xa[ia], ya[ia], Xa[ib], ya[ib]

def load_data():
    base = Path(cfg.DATA_DIR)
    X = np.load(base / "X.npy").astype(np.float32)     # [N,64,64]=[N,time,Doppler]
    y = np.load(base / "y.npy").astype(np.int64)
    assert X.ndim == 3, f"expected [N,64,64], got {X.shape}"
    Xtv, ytv, Xte, yte = _strat_split(X, y, 0.20, 0)
    Xtr, ytr, Xva, yva = _strat_split(Xtv, ytv, cfg.VAL_FRAC/0.80, 0)
    out = {}
    for tname in cfg.INPUT_CHOICES:
        if tname == "log1p":
            f = lambda a: np.log1p(np.clip(a, 0.0, None))
            tr, va, te = f(Xtr), f(Xva), f(Xte)
        else:
            tr, va, te = Xtr, Xva, Xte
        mu, sd = float(tr.mean()), float(tr.std() + 1e-6)
        nz = lambda a: torch.as_tensor((a - mu)/sd).float()
        out[tname] = (nz(tr), torch.as_tensor(ytr).long(),
                      nz(va), torch.as_tensor(yva).long(),
                      nz(te), torch.as_tensor(yte).long())
    C = int(y.max() + 1); Fin = X.shape[-1]
    return out, C, Fin

DATA, C, Fin = load_data()
n_tr = len(DATA[cfg.INPUT_CHOICES[0]][0]); n_va = len(DATA[cfg.INPUT_CHOICES[0]][2]); n_te = len(DATA[cfg.INPUT_CHOICES[0]][4])
print(f"[DATA] train={n_tr} val={n_va} test={n_te}  C={C}  F={Fin}  chance={1/C:.3f}", flush=True)

# ══════════════════════════════════════════════════════════════════════════════
# Metrics / eval / bootstrap  (verbatim from the ablation)
# ══════════════════════════════════════════════════════════════════════════════
def accuracy(p, t, C=None): return float((p == t).mean())
def macro_f1(p, t, C):
    f1s = []
    for c in range(C):
        tp = int(np.sum((p == c) & (t == c))); fp = int(np.sum((p == c) & (t != c)))
        fn = int(np.sum((p != c) & (t == c)))
        pr = tp/(tp+fp) if (tp+fp) else 0.0; rc = tp/(tp+fn) if (tp+fn) else 0.0
        f1s.append(2*pr*rc/(pr+rc) if (pr+rc) else 0.0)
    return float(np.mean(f1s)), [round(float(x), 4) for x in f1s]
METRICS = {"accuracy": lambda p, t, C: accuracy(p, t),
           "macro_f1": lambda p, t, C: macro_f1(p, t, C)[0]}

@torch.no_grad()
def evaluate(model, X, Y, batch):
    model.eval(); preds = []
    for i in range(0, len(X), batch):
        preds.append(model(X[i:i+batch].to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy(), Y.numpy()

def firing_mean(model, xb):
    fr = model.firing_rates(xb) if isinstance(model, VerticalNet) else model.path_firing(xb)
    return float(np.mean(fr))

def paired_bootstrap(pa, pb, trues, mfn, C, nboot=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(trues)
    base = mfn(pa, trues, C) - mfn(pb, trues, C); boots = np.empty(nboot)
    for k in range(nboot):
        i = rng.integers(0, n, n)
        boots[k] = mfn(pa[i], trues[i], C) - mfn(pb[i], trues[i], C)   # paired resample
    return float(base), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

# ══════════════════════════════════════════════════════════════════════════════
# Model builder + train/eval  (explicit hyperparams; val-smoothed selection)
# ══════════════════════════════════════════════════════════════════════════════
def build_model(arch, W, L, beta):
    if arch == "doppler":
        nk = {"Doppler-LIF": {"beta": beta}}
        return VerticalNet(Fin, C, ["Doppler-LIF"]*L, W, nk, "none")
    if arch == "lif":
        nk = {"Vanilla-LIF": {"beta": beta}}
        return VerticalNet(Fin, C, ["Vanilla-LIF"]*L, W, nk, "none")
    if arch == "d2":
        nk = {t: {"beta": beta} for t in D2_PATHS}
        return PathBankNet(Fin, C, D2_PATHS, W, L, "concat", nk, "none", True)
    raise KeyError(arch)

def make_sched(opt, schedule, max_epochs, warmup=5):
    if schedule != "cosine": return None
    def fn(ep):
        if ep < warmup: return (ep + 1) / warmup
        prog = (ep - warmup) / max(1, max_epochs - warmup)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def train_eval(arch, p, seed, ckpt_tag, trial=None, verbose=False):
    """Returns dict: f1, acc, firing, params, best_epoch, preds, trues (test @ best smoothed-val)."""
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    if SURROGATE is not None:
        SURROGATE.scale = p["surr"]                               # global surrogate slope

    Xtr, ytr, Xva, yva, Xte, yte = DATA[p["input"]]
    fire_batch = Xtr[:256].to(device)
    model = build_model(arch, p["W"], p["L"], p["beta"]).to(device)
    calibrate_thresholds(model, fire_batch, p["fire"])            # config A: init only
    n_params = count_params(model)
    opt = torch.optim.Adam(model.parameters(), lr=p["lr"], weight_decay=cfg.WEIGHT_DECAY)
    sched = make_sched(opt, p["sched"], cfg.MAX_EPOCHS)

    ckpt = CKPT_DIR / f"{ckpt_tag}.pt"
    start, m = 0, {}
    if ckpt.exists():
        ck = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        if sched is not None and ck.get("sched"): sched.load_state_dict(ck["sched"])
        start, m = ck["epoch"] + 1, ck["m"]

    sel = METRICS[cfg.SELECT_METRIC]
    val_hist = m.get("val_hist", []); best_val = m.get("best_val", -1.0)
    best_epoch = m.get("best_epoch", -1); pc = m.get("pc", 0); best = m.get("best", None)
    batch = cfg.BATCH

    use_bar = cfg.USE_TQDM and _HAS_TQDM
    epochs = range(start, cfg.MAX_EPOCHS)
    bar = tqdm(epochs, desc=ckpt_tag, leave=False, dynamic_ncols=True,
               initial=start, total=cfg.MAX_EPOCHS) if use_bar else epochs
    for ep in bar:
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), batch):
            b = perm[i:i+batch]
            xb, yb = Xtr[b].to(device), ytr[b].to(device)
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(xb), yb).backward(); opt.step()
        if sched is not None: sched.step()

        vp, vt = evaluate(model, Xva, yva, batch); v_raw = sel(vp, vt, C)
        val_hist.append(v_raw)
        v_smooth = float(np.mean(val_hist[-cfg.VAL_SMOOTH:]))
        tp, tt = evaluate(model, Xte, yte, batch)
        t_acc = accuracy(tp, tt); t_f1, t_f1c = macro_f1(tp, tt, C)
        improved = v_smooth > best_val + 1e-4
        if improved:
            best_val, best_epoch, pc = v_smooth, ep, 0
            best = {"acc": t_acc, "f1": t_f1, "f1_per_class": t_f1c,
                    "firing": firing_mean(model, fire_batch),
                    "preds": tp.tolist(), "trues": tt.tolist()}
        else:
            pc += 1
        torch.save({"epoch": ep, "model": copy.deepcopy(model.state_dict()), "opt": opt.state_dict(),
                    "sched": sched.state_dict() if sched is not None else None,
                    "m": {"val_hist": val_hist, "best_val": best_val, "best_epoch": best_epoch,
                          "pc": pc, "best": best}}, ckpt)
        cur_fire = best["firing"] if best else 0.0
        if use_bar:
            bar.set_postfix(val=f"{v_smooth:.3f}", f1=f"{t_f1:.3f}", fire=f"{cur_fire:.2f}", pat=pc)
        elif (ep % cfg.HEARTBEAT_EVERY == 0) or improved:
            print(f"      ep{ep:>3}/{cfg.MAX_EPOCHS}  val={v_smooth:.4f}  f1={t_f1:.4f}  "
                  f"fire={cur_fire:.3f}  pat={pc}{'  *best' if improved else ''}", flush=True)
        # conservative absolute-floor pruning (HPO only; safe vs late convergence)
        if trial is not None and ep == cfg.PRUNE_EPOCH and v_smooth < cfg.PRUNE_FLOOR:
            if use_bar: bar.close()
            raise optuna.TrialPruned()
        if pc >= cfg.PATIENCE:
            break
    if use_bar:
        bar.close()
    best["params"] = n_params; best["best_epoch"] = best_epoch
    return best

# ══════════════════════════════════════════════════════════════════════════════
# Optuna objective  (per architecture; multi-objective: maximize F1, minimize firing)
# ══════════════════════════════════════════════════════════════════════════════
def make_objective(arch):
    def objective(trial):
        p = dict(
            W     = trial.suggest_categorical("W", cfg.W_CHOICES),
            L     = trial.suggest_categorical("L", cfg.L_CHOICES),
            beta  = trial.suggest_categorical("beta", cfg.BETA_CHOICES),
            lr    = trial.suggest_float("lr", cfg.LR_LOW, cfg.LR_HIGH, log=True),
            sched = trial.suggest_categorical("sched", cfg.SCHED_CHOICES),
            surr  = trial.suggest_categorical("surr", cfg.SURR_CHOICES),
            fire  = trial.suggest_categorical("fire", cfg.FIRE_CHOICES),
            input = trial.suggest_categorical("input", cfg.INPUT_CHOICES),
        )
        tag = f"{arch}_t{trial.number}_s{cfg.HPO_SEED}"
        print(f"\n[{arch} trial {trial.number}/{cfg.N_TRIALS}] W={p['W']} L={p['L']} beta={p['beta']} "
              f"lr={p['lr']:.1e} {p['sched']} surr={p['surr']} fire={p['fire']} {p['input']}", flush=True)
        r = train_eval(arch, p, cfg.HPO_SEED, tag, trial=trial)
        print(f"   -> trial {trial.number} done: F1={r['f1']:.4f}  acc={r['acc']:.4f}  "
              f"fire={r['firing']:.3f}  @ep{r['best_epoch']}  params={r['params']}", flush=True)
        trial.set_user_attr("params", r["params"])
        trial.set_user_attr("best_epoch", r["best_epoch"])
        trial.set_user_attr("acc", r["acc"])
        return r["f1"], r["firing"]            # (maximize, minimize)
    return objective

def run_study(arch):
    study = optuna.create_study(
        study_name=f"hpo_{arch}", storage=STORAGE, load_if_exists=True,
        directions=["maximize", "minimize"],
        sampler=optuna.samplers.TPESampler(multivariate=True, seed=cfg.HPO_SEED))
    done = len([t for t in study.trials if t.state.name in ("COMPLETE", "PRUNED")])
    todo = max(0, cfg.N_TRIALS - done)
    print(f"\n[STUDY {arch}] {done}/{cfg.N_TRIALS} done, running {todo} more", flush=True)
    if todo:
        study.optimize(make_objective(arch), n_trials=todo, show_progress_bar=False)
    return study

# ══════════════════════════════════════════════════════════════════════════════
# Run all three studies, extract Pareto, select configs, re-evaluate on 3 seeds
# ══════════════════════════════════════════════════════════════════════════════
def select_configs(study):
    """From the Pareto front pick (best_f1) and a firing-efficient (high-F1, min-firing) config."""
    pf = [t for t in study.best_trials]
    if not pf:
        comp = [t for t in study.trials if t.state.name == "COMPLETE"]
        pf = sorted(comp, key=lambda t: t.values[0], reverse=True)[:1]
    best_f1 = max(pf, key=lambda t: t.values[0])
    f1max = best_f1.values[0]
    near = [t for t in pf if t.values[0] >= f1max - 0.01]
    efficient = min(near, key=lambda t: t.values[1])
    out = {"best_f1": best_f1.params}
    if efficient.number != best_f1.number:
        out["efficient"] = efficient.params
    return out, pf

REEVAL_PATH = RUN_DIR / "reeval.json"
reeval = json.loads(REEVAL_PATH.read_text()) if REEVAL_PATH.exists() else {}

studies, pareto = {}, {}
for arch in cfg.ARCHS:
    studies[arch] = run_study(arch)

print("\n" + "═"*78 + "\n PARETO FRONTS  (F1 ↑, firing ↓, params)\n" + "═"*78)
selected = {}
for arch in cfg.ARCHS:
    cfgs, pf = select_configs(studies[arch]); selected[arch] = cfgs; pareto[arch] = []
    print(f"\n[{arch}]  ({len(pf)} Pareto points)")
    for t in sorted(pf, key=lambda x: x.values[0], reverse=True):
        pr = t.user_attrs.get("params", -1)
        pareto[arch].append({"f1": t.values[0], "firing": t.values[1], "params": pr, "params_cfg": t.params})
        print(f"   F1={t.values[0]:.4f}  fire={t.values[1]:.3f}  params={pr:>7}  "
              f"W={t.params['W']} L={t.params['L']} beta={t.params['beta']} lr={t.params['lr']:.1e} "
              f"sched={t.params['sched']} surr={t.params['surr']} fire={t.params['fire']} in={t.params['input']}")

# ── re-evaluate selected configs on 3 seeds ───────────────────────────────────
print("\n" + "═"*78 + "\n RE-EVAL selected configs on seeds " + str(cfg.REEVAL_SEEDS) + "\n" + "═"*78)
for arch in cfg.ARCHS:
    for cname, params in selected[arch].items():
        for seed in cfg.REEVAL_SEEDS:
            key = f"{arch}:{cname}:{seed}"
            if key in reeval:
                continue
            p = dict(W=params["W"], L=params["L"], beta=params["beta"], lr=params["lr"],
                     sched=params["sched"], surr=params["surr"], fire=params["fire"], input=params["input"])
            tag = f"reeval_{arch}_{cname}_s{seed}"
            print(f"\n[re-eval {arch}:{cname} seed {seed}] W={params['W']} L={params['L']} "
                  f"beta={params['beta']} {params['sched']} fire={params['fire']}", flush=True)
            r = train_eval(arch, p, seed, tag)
            reeval[key] = {"f1": r["f1"], "acc": r["acc"], "firing": r["firing"],
                           "params": r["params"], "best_epoch": r["best_epoch"],
                           "preds": r["preds"], "trues": r["trues"]}
            REEVAL_PATH.write_text(json.dumps(reeval))
            print(f"   {key:<34} F1={r['f1']:.4f} acc={r['acc']:.4f} fire={r['firing']:.3f} params={r['params']}", flush=True)

# ── aggregate re-eval + paired bootstrap head-to-head (best_f1 configs) ────────
def agg(arch, cname):
    ks = [f"{arch}:{cname}:{s}" for s in cfg.REEVAL_SEEDS if f"{arch}:{cname}:{s}" in reeval]
    f1s = [reeval[k]["f1"] for k in ks]; firs = [reeval[k]["firing"] for k in ks]
    accs = [reeval[k]["acc"] for k in ks]; pr = reeval[ks[0]]["params"]
    return ks, f1s, firs, accs, pr

print("\n" + "═"*78 + "\n FINAL  (best_f1 config, mean±std over re-eval seeds)\n" + "═"*78)
final = {}
for arch in cfg.ARCHS:
    ks, f1s, firs, accs, pr = agg(arch, "best_f1")
    final[arch] = {"f1_mean": float(np.mean(f1s)), "f1_std": float(np.std(f1s)),
                   "acc_mean": float(np.mean(accs)), "firing_mean": float(np.mean(firs)),
                   "params": pr, "keys": ks}
    print(f"   {arch:<9} F1 {np.mean(f1s):.4f}±{np.std(f1s):.4f}  acc {np.mean(accs):.4f}  "
          f"fire {np.mean(firs):.3f}  params {pr}")

print("\n[BOOTSTRAP] paired on test, per re-eval seed (consistent = same sign all seeds)")
boot = {}
pairs = [("doppler", "lif"), ("d2", "lif"), ("doppler", "d2")]
for a, b in pairs:
    if a not in cfg.ARCHS or b not in cfg.ARCHS: continue
    for label, mfn in [("F1", METRICS["macro_f1"]), ("acc", METRICS["accuracy"])]:
        per_seed = []
        for s in cfg.REEVAL_SEEDS:
            ka, kb = f"{a}:best_f1:{s}", f"{b}:best_f1:{s}"
            if ka in reeval and kb in reeval:
                pa = np.array(reeval[ka]["preds"]); pb = np.array(reeval[kb]["preds"])
                tr = np.array(reeval[ka]["trues"])
                per_seed.append(paired_bootstrap(pa, pb, tr, mfn, C))
        if not per_seed: continue
        mean = float(np.mean([x[0] for x in per_seed]))
        cis = [(round(lo, 3), round(hi, 3)) for _, lo, hi in per_seed]
        consistent = all(lo > 0 for _, lo, _ in per_seed) or all(hi < 0 for _, _, hi in per_seed)
        boot[f"{a}-{b}[{label}]"] = {"mean": mean, "cis": cis, "consistent": consistent}
        print(f"   {a} - {b} [{label}]: mean {mean:+.4f}  CIs {cis}  consistent={consistent}")

# ── save ──────────────────────────────────────────────────────────────────────
summary = {"run_dir": str(RUN_DIR), "archs": cfg.ARCHS, "n_trials": cfg.N_TRIALS,
           "reeval_seeds": cfg.REEVAL_SEEDS, "search_space": {
               "W": cfg.W_CHOICES, "L": cfg.L_CHOICES, "beta": cfg.BETA_CHOICES,
               "lr": [cfg.LR_LOW, cfg.LR_HIGH], "sched": cfg.SCHED_CHOICES, "surr": cfg.SURR_CHOICES,
               "fire": cfg.FIRE_CHOICES, "input": cfg.INPUT_CHOICES},
           "selected": selected, "pareto": pareto, "final": final, "bootstrap": boot,
           "timestamp": datetime.now().isoformat()}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"\n[DONE] summary -> {RUN_DIR / 'summary.json'}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DIAT-uSAT  ·  BEST-MODEL SEARCH  (application phase, separate from the ablation)
# Multi-objective HPO:  maximize test macro-F1   ·   minimize test mean-firing
# Candidates:  Doppler-homog   ·   D2 path-bank (concat)   ·   Vanilla-LIF (BASELINE)
#
# Design (locked):
#   - Baseline = matched Vanilla-LIF: same BaseNeuron interface, same FastSigmoid
#     surrogate, plain leaky-integrate-fire dynamics, ZERO learnable neuron params.
#     Isolates the spectro-temporal inductive bias, not the implementation.
#   - Selection machinery is IDENTICAL to the ablation: per-epoch threshold
#     calibration at init (config A), early-stop on SMOOTHED validation macro-F1,
#     test measured ONCE at the best-smoothed-val epoch (no test peeking).
#   - HPO runs on a single seed (fast); the Pareto-selected configs are then
#     RE-EVALUATED on 3 seeds for the final report + paired bootstrap.
#   - Pruning is a conservative ABSOLUTE floor (won't kill late-convergers:
#     in the ablation, strong configs only converged at ep 130-190).
#   - param-count is logged per trial as a third column (FPGA/area budget).
#
# Single paste-and-run Colab cell. SQLite storage → resumes across sessions.
# ══════════════════════════════════════════════════════════════════════════════
import os, json, math, copy, time, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

# ── Colab vs sandbox ──────────────────────────────────────────────────────────
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ────────────────────────────── CONFIG (overridable) ──────────────────────────
class HPO:
    # search space (8 high-impact dims; wd/batch held at sane defaults for the main search)
    W_CHOICES        = [96, 128, 192, 256, 384]     # width — deployable cap, not 2048
    L_CHOICES        = [3, 4, 5, 6]                  # depth
    BETA_CHOICES     = [0.85, 0.90, 0.95, 0.97, 0.99]  # membrane decay (integration window)
    LR_LOW, LR_HIGH  = 3e-4, 4e-3                    # log-uniform
    SCHED_CHOICES    = ["none", "cosine"]           # none | warmup+cosine
    SURR_CHOICES     = [10.0, 25.0, 50.0]           # FastSigmoid surrogate slope (global attr)
    FIRE_CHOICES     = [0.10, 0.15, 0.20]           # calibration target -> threshold; also moves cost axis
    INPUT_CHOICES    = ["plain", "log1p"]           # input transform (radar power dynamic range)
    WEIGHT_DECAY     = 0.0                           # fixed for main search
    BATCH            = 128                           # fixed for main search

    # objectives / training
    MAX_EPOCHS       = 200
    PATIENCE         = 20
    VAL_SMOOTH       = 5
    VAL_FRAC         = 0.15
    SELECT_METRIC    = "macro_f1"                    # consistent with ablation

    # budget
    N_TRIALS         = 55                            # per architecture
    HPO_SEED         = 42                            # single seed for the search
    REEVAL_SEEDS     = [42, 123, 999]                # multi-seed re-eval of selected configs
    ARCHS            = ["doppler", "d2", "lif"]      # lif = baseline

    # visibility (so the run is not a black box)
    USE_TQDM         = True                          # live per-epoch progress bar inside each trial
    HEARTBEAT_EVERY  = 5                             # if no tqdm: print a status line every N epochs

    # conservative absolute-floor pruning (safe vs late convergence)
    PRUNE_EPOCH      = 30
    PRUNE_FLOOR      = 0.30                          # 6-class chance=0.167; collapse (STFT) stays ~0.15-0.18

    # paths
    if IN_COLAB:
        PROJECT  = "/content/drive/MyDrive/Research/NISAC_DeepHetero"
        DATA_DIR = "/content/drive/MyDrive/NISAC/data/DIAT_uSAT/processed"
    else:
        PROJECT  = "/home/claude/hpo_run"
        DATA_DIR = "/home/claude/syn_diat"

cfg = HPO()

# ── ACTIVE RUN SCOPE ──────────────────────────────────────────────────────────
# D2 is already dominated: its 8 trials show the whole D2 front sits at LOWER F1
# and HIGHER firing than Doppler's, even though D2 is NOT capacity-matched here
# (5x path params at a given W). Finishing its 55 (~2-3 h/trial on T4) buys nothing.
# Doppler is done (55/55 -> resumes at 0-todo). The crux left is LIF (the baseline),
# which is fast (homogeneous, Doppler-speed). TPE locked the good region by ~trial
# 17, so 30 trials is plenty. Set RUN_ID unchanged so everything resumes in place.
cfg.ARCHS    = ["doppler", "lif", "plif"]
cfg.N_TRIALS = 30
cfg.REEVAL_SEEDS = [42, 123, 999, 7, 2024]   # 5-seed headline; doppler/lif resume (3 cached + 2 new)

# ── reduced sandbox overrides (set REDUCED=1 in env for a fast end-to-end check) ─
if os.environ.get("REDUCED") == "1":
    cfg.W_CHOICES   = [16, 24, 32]
    cfg.L_CHOICES   = [2, 3]
    cfg.MAX_EPOCHS  = 6
    cfg.PATIENCE    = 4
    cfg.VAL_SMOOTH  = 2
    cfg.N_TRIALS    = 3
    cfg.REEVAL_SEEDS = [42]
    cfg.PRUNE_EPOCH = 99          # don't prune in the tiny check
    cfg.ARCHS       = ["doppler", "lif", "plif", "d2"]

# ── Colab-only setup ──────────────────────────────────────────────────────────
if IN_COLAB:
    os.system("pip install -q git+https://github.com/DrCanD/dikmen-spiking-neurons.git optuna")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import optuna
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False
import dikmen_neurons as D
from dikmen_neurons import BaseNeuron, spike_hard, NeuronRegistry

# FastSigmoidSurrogate lives in the submodule dikmen_neurons.base (NOT top-level).
# spike_hard() calls it from its own module globals, so grab the exact class object
# there; setting .scale on it is what backward() reads (class-attribute lookup).
def _locate_surrogate():
    c = getattr(D, "FastSigmoidSurrogate", None)
    if c is not None:
        return c
    sh = getattr(D, "spike_hard", None)
    if sh is not None and hasattr(sh, "__globals__"):
        c = sh.__globals__.get("FastSigmoidSurrogate")
        if c is not None:
            return c
    try:
        import pkgutil, importlib
        for _, name, _ in pkgutil.walk_packages(D.__path__, D.__name__ + "."):
            m = importlib.import_module(name)
            if hasattr(m, "FastSigmoidSurrogate"):
                return getattr(m, "FastSigmoidSurrogate")
    except Exception:
        pass
    return None

SURROGATE = _locate_surrogate()
if SURROGATE is None:
    print("[WARN] FastSigmoidSurrogate not found; surrogate-slope tuning disabled (package default).")

device = "cuda" if torch.cuda.is_available() else "cpu"
Path(cfg.PROJECT).mkdir(parents=True, exist_ok=True)
RUN_ID  = os.environ.get("RUN_ID", "hpo_main")   # STABLE across re-runs -> resume; change for a new experiment
RUN_DIR = Path(cfg.PROJECT) / "hpo" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = RUN_DIR / "ckpt"; CKPT_DIR.mkdir(exist_ok=True)
STORAGE  = f"sqlite:///{RUN_DIR / 'studies.db'}"
print(f"[HW] device={device}" + (f"  gpu={torch.cuda.get_device_name(0)}" if device == 'cuda' else ""))
print(f"[RUN] {RUN_DIR}", flush=True)

# ══════════════════════════════════════════════════════════════════════════════
# Matched Vanilla-LIF baseline  (registered into the package registry)
# ══════════════════════════════════════════════════════════════════════════════
class VanillaLIF(BaseNeuron):
    _family = "lif"
    _description = "Standard leaky integrate-and-fire baseline (no resonance/freq-selectivity)."
    def single_step(self, x_t, state):
        mem = self.beta * state["mem"] + x_t
        spk = spike_hard(mem, self.threshold)
        mem = mem * (1.0 - spk)
        return spk, {"mem": mem}
NeuronRegistry._all["Vanilla-LIF"] = VanillaLIF

# Parametric-LIF baseline (Fang et al. PLIF): learnable per-neuron decay (tau), same
# dynamics as vanilla LIF otherwise. Isolates "generic learnable adaptivity" (W params)
# from Doppler's specific resonance/freq-selectivity (3W params). Ladder: 0 -> W -> 3W.
class ParametricLIF(BaseNeuron):
    _family = "plif"
    _description = "Parametric LIF: learnable per-neuron membrane decay (no resonance)."
    def __init__(self, size, beta=0.95, threshold=1.0):
        super().__init__(size, beta=beta, threshold=threshold)
        b = min(max(float(beta), 1e-3), 1 - 1e-3)
        self.beta_logit = nn.Parameter(torch.full((size,), math.log(b / (1 - b))))
    def single_step(self, x_t, state):
        beta = torch.sigmoid(self.beta_logit)
        mem = beta * state["mem"] + x_t
        spk = spike_hard(mem, self.threshold)
        mem = mem * (1.0 - spk)
        return spk, {"mem": mem}
NeuronRegistry._all["Parametric-LIF"] = ParametricLIF

D2_PATHS = ["Doppler-LIF", "Chirp-LIF", "STFT-IF", "Dual-tau-LIF", "CrossInhib-LIF"]

# ══════════════════════════════════════════════════════════════════════════════
# Deep bodies (inline) — identical to the ablation harness
# ══════════════════════════════════════════════════════════════════════════════
def _make_norm(kind, width):
    if kind == "layer": return nn.LayerNorm(width)
    if kind == "batch": return nn.BatchNorm1d(width)
    return nn.Identity()                      # PURE SNN default

class FeatureStack(nn.Module):
    def __init__(self, in_features, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        neuron_kwargs = neuron_kwargs or {}
        dims = [in_features] + [width] * len(layer_types)
        self.projs = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(layer_types))])
        self.norms = nn.ModuleList([_make_norm(norm, width) for _ in layer_types])
        self.neurons = nn.ModuleList(
            [NeuronRegistry.create(t, size=width, **neuron_kwargs.get(t, {})) for t in layer_types])
        self.width = width
    def forward(self, x):
        h = x
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); h = spk
        return h.mean(dim=1), h
    def layer_firing(self, x):
        h, rates = x, []
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); rates.append(spk.float().mean().item()); h = spk
        return rates

class VerticalNet(nn.Module):
    def __init__(self, in_features, n_classes, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        self.stack = FeatureStack(in_features, layer_types, width, neuron_kwargs, norm)
        self.readout = nn.Linear(width, n_classes)
    def forward(self, x):
        feat, _ = self.stack(x); return self.readout(feat)
    def firing_rates(self, x): return self.stack.layer_firing(x)

class PathBankNet(nn.Module):
    def __init__(self, in_features, n_classes, path_types, width, n_layers,
                 fusion="concat", neuron_kwargs=None, norm="none", shared_stem=True):
        super().__init__()
        if shared_stem:
            self.stem = nn.Linear(in_features, width); path_in = width
        else:
            self.stem = None; path_in = in_features
        self.paths = nn.ModuleList(
            [FeatureStack(path_in, [t]*n_layers, width, neuron_kwargs, norm) for t in path_types])
        self.fusion = fusion; self.width = width
        feat_dim = width * len(path_types)
        self.readout = nn.Linear(feat_dim, n_classes)
    def forward(self, x):
        if self.stem is not None:
            B, T, d = x.shape; x = self.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        feats = [p(x)[0] for p in self.paths]
        return self.readout(torch.cat(feats, dim=-1))
    def path_firing(self, x):
        if self.stem is not None:
            B, T, d = x.shape; x = self.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        return [p(x)[1].float().mean().item() for p in self.paths]

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def _calibrate_stack(stack, h, target, iters=30):
    for proj, norm, neuron in zip(stack.projs, stack.norms, stack.neurons):
        B, T, d = h.shape
        z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
        a, b = 1e-3, 1e5
        for _ in range(iters):
            mid = (a*b)**0.5; neuron.threshold = mid
            r = neuron(z)[0].float().mean().item()
            if r > target: a = mid
            else:          b = mid
        neuron.threshold = (a*b)**0.5
        h = neuron(z)[0]
    return stack

@torch.no_grad()
def calibrate_thresholds(model, x, target):
    if isinstance(model, VerticalNet):
        _calibrate_stack(model.stack, x, target)
    else:
        h = x
        if model.stem is not None:
            B, T, d = x.shape; h = model.stem(x.reshape(B*T, d)).reshape(B, T, -1)
        for p in model.paths: _calibrate_stack(p, h, target)
    return model

# ══════════════════════════════════════════════════════════════════════════════
# Data  (precompute plain + log1p standardized splits once; train-stat standardize)
# ══════════════════════════════════════════════════════════════════════════════
def _strat_split(Xa, ya, frac, seed):
    try:
        from sklearn.model_selection import train_test_split
        A, B, ya2, yb2 = train_test_split(Xa, ya, test_size=frac, stratify=ya, random_state=seed)
        return A, ya2, B, yb2
    except Exception:
        rng = np.random.default_rng(seed); ia, ib = [], []
        for c in np.unique(ya):
            ci = np.where(ya == c)[0]; rng.shuffle(ci); k = int((1-frac)*len(ci))
            ia += list(ci[:k]); ib += list(ci[k:])
        return Xa[ia], ya[ia], Xa[ib], ya[ib]

def load_data():
    base = Path(cfg.DATA_DIR)
    X = np.load(base / "X.npy").astype(np.float32)     # [N,64,64]=[N,time,Doppler]
    y = np.load(base / "y.npy").astype(np.int64)
    assert X.ndim == 3, f"expected [N,64,64], got {X.shape}"
    Xtv, ytv, Xte, yte = _strat_split(X, y, 0.20, 0)
    Xtr, ytr, Xva, yva = _strat_split(Xtv, ytv, cfg.VAL_FRAC/0.80, 0)
    out = {}
    for tname in cfg.INPUT_CHOICES:
        if tname == "log1p":
            f = lambda a: np.log1p(np.clip(a, 0.0, None))
            tr, va, te = f(Xtr), f(Xva), f(Xte)
        else:
            tr, va, te = Xtr, Xva, Xte
        mu, sd = float(tr.mean()), float(tr.std() + 1e-6)
        nz = lambda a: torch.as_tensor((a - mu)/sd).float()
        out[tname] = (nz(tr), torch.as_tensor(ytr).long(),
                      nz(va), torch.as_tensor(yva).long(),
                      nz(te), torch.as_tensor(yte).long())
    C = int(y.max() + 1); Fin = X.shape[-1]
    return out, C, Fin

DATA, C, Fin = load_data()
n_tr = len(DATA[cfg.INPUT_CHOICES[0]][0]); n_va = len(DATA[cfg.INPUT_CHOICES[0]][2]); n_te = len(DATA[cfg.INPUT_CHOICES[0]][4])
print(f"[DATA] train={n_tr} val={n_va} test={n_te}  C={C}  F={Fin}  chance={1/C:.3f}", flush=True)

# ══════════════════════════════════════════════════════════════════════════════
# Metrics / eval / bootstrap  (verbatim from the ablation)
# ══════════════════════════════════════════════════════════════════════════════
def accuracy(p, t, C=None): return float((p == t).mean())
def macro_f1(p, t, C):
    f1s = []
    for c in range(C):
        tp = int(np.sum((p == c) & (t == c))); fp = int(np.sum((p == c) & (t != c)))
        fn = int(np.sum((p != c) & (t == c)))
        pr = tp/(tp+fp) if (tp+fp) else 0.0; rc = tp/(tp+fn) if (tp+fn) else 0.0
        f1s.append(2*pr*rc/(pr+rc) if (pr+rc) else 0.0)
    return float(np.mean(f1s)), [round(float(x), 4) for x in f1s]
METRICS = {"accuracy": lambda p, t, C: accuracy(p, t),
           "macro_f1": lambda p, t, C: macro_f1(p, t, C)[0]}

@torch.no_grad()
def evaluate(model, X, Y, batch):
    model.eval(); preds = []
    for i in range(0, len(X), batch):
        preds.append(model(X[i:i+batch].to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy(), Y.numpy()

def firing_mean(model, xb):
    fr = model.firing_rates(xb) if isinstance(model, VerticalNet) else model.path_firing(xb)
    return float(np.mean(fr))

def paired_bootstrap(pa, pb, trues, mfn, C, nboot=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(trues)
    base = mfn(pa, trues, C) - mfn(pb, trues, C); boots = np.empty(nboot)
    for k in range(nboot):
        i = rng.integers(0, n, n)
        boots[k] = mfn(pa[i], trues[i], C) - mfn(pb[i], trues[i], C)   # paired resample
    return float(base), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

# ══════════════════════════════════════════════════════════════════════════════
# Model builder + train/eval  (explicit hyperparams; val-smoothed selection)
# ══════════════════════════════════════════════════════════════════════════════
def build_model(arch, W, L, beta):
    if arch == "doppler":
        nk = {"Doppler-LIF": {"beta": beta}}
        return VerticalNet(Fin, C, ["Doppler-LIF"]*L, W, nk, "none")
    if arch == "lif":
        nk = {"Vanilla-LIF": {"beta": beta}}
        return VerticalNet(Fin, C, ["Vanilla-LIF"]*L, W, nk, "none")
    if arch == "plif":
        nk = {"Parametric-LIF": {"beta": beta}}
        return VerticalNet(Fin, C, ["Parametric-LIF"]*L, W, nk, "none")
    if arch == "d2":
        nk = {t: {"beta": beta} for t in D2_PATHS}
        return PathBankNet(Fin, C, D2_PATHS, W, L, "concat", nk, "none", True)
    raise KeyError(arch)

def make_sched(opt, schedule, max_epochs, warmup=5):
    if schedule != "cosine": return None
    def fn(ep):
        if ep < warmup: return (ep + 1) / warmup
        prog = (ep - warmup) / max(1, max_epochs - warmup)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def train_eval(arch, p, seed, ckpt_tag, trial=None, verbose=False):
    """Returns dict: f1, acc, firing, params, best_epoch, preds, trues (test @ best smoothed-val)."""
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    if SURROGATE is not None:
        SURROGATE.scale = p["surr"]                               # global surrogate slope

    Xtr, ytr, Xva, yva, Xte, yte = DATA[p["input"]]
    fire_batch = Xtr[:256].to(device)
    model = build_model(arch, p["W"], p["L"], p["beta"]).to(device)
    calibrate_thresholds(model, fire_batch, p["fire"])            # config A: init only
    n_params = count_params(model)
    opt = torch.optim.Adam(model.parameters(), lr=p["lr"], weight_decay=cfg.WEIGHT_DECAY)
    sched = make_sched(opt, p["sched"], cfg.MAX_EPOCHS)

    ckpt = CKPT_DIR / f"{ckpt_tag}.pt"
    start, m = 0, {}
    if ckpt.exists():
        ck = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        if sched is not None and ck.get("sched"): sched.load_state_dict(ck["sched"])
        start, m = ck["epoch"] + 1, ck["m"]

    sel = METRICS[cfg.SELECT_METRIC]
    val_hist = m.get("val_hist", []); best_val = m.get("best_val", -1.0)
    best_epoch = m.get("best_epoch", -1); pc = m.get("pc", 0); best = m.get("best", None)
    batch = cfg.BATCH

    use_bar = cfg.USE_TQDM and _HAS_TQDM
    epochs = range(start, cfg.MAX_EPOCHS)
    bar = tqdm(epochs, desc=ckpt_tag, leave=False, dynamic_ncols=True,
               initial=start, total=cfg.MAX_EPOCHS) if use_bar else epochs
    for ep in bar:
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), batch):
            b = perm[i:i+batch]
            xb, yb = Xtr[b].to(device), ytr[b].to(device)
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(xb), yb).backward(); opt.step()
        if sched is not None: sched.step()

        vp, vt = evaluate(model, Xva, yva, batch); v_raw = sel(vp, vt, C)
        val_hist.append(v_raw)
        v_smooth = float(np.mean(val_hist[-cfg.VAL_SMOOTH:]))
        tp, tt = evaluate(model, Xte, yte, batch)
        t_acc = accuracy(tp, tt); t_f1, t_f1c = macro_f1(tp, tt, C)
        improved = v_smooth > best_val + 1e-4
        if improved:
            best_val, best_epoch, pc = v_smooth, ep, 0
            best = {"acc": t_acc, "f1": t_f1, "f1_per_class": t_f1c,
                    "firing": firing_mean(model, fire_batch),
                    "preds": tp.tolist(), "trues": tt.tolist()}
        else:
            pc += 1
        torch.save({"epoch": ep, "model": copy.deepcopy(model.state_dict()), "opt": opt.state_dict(),
                    "sched": sched.state_dict() if sched is not None else None,
                    "m": {"val_hist": val_hist, "best_val": best_val, "best_epoch": best_epoch,
                          "pc": pc, "best": best}}, ckpt)
        cur_fire = best["firing"] if best else 0.0
        if use_bar:
            bar.set_postfix(val=f"{v_smooth:.3f}", f1=f"{t_f1:.3f}", fire=f"{cur_fire:.2f}", pat=pc)
        elif (ep % cfg.HEARTBEAT_EVERY == 0) or improved:
            print(f"      ep{ep:>3}/{cfg.MAX_EPOCHS}  val={v_smooth:.4f}  f1={t_f1:.4f}  "
                  f"fire={cur_fire:.3f}  pat={pc}{'  *best' if improved else ''}", flush=True)
        # conservative absolute-floor pruning (HPO only; safe vs late convergence)
        if trial is not None and ep == cfg.PRUNE_EPOCH and v_smooth < cfg.PRUNE_FLOOR:
            if use_bar: bar.close()
            raise optuna.TrialPruned()
        if pc >= cfg.PATIENCE:
            break
    if use_bar:
        bar.close()
    best["params"] = n_params; best["best_epoch"] = best_epoch
    return best

# ══════════════════════════════════════════════════════════════════════════════
# Optuna objective  (per architecture; multi-objective: maximize F1, minimize firing)
# ══════════════════════════════════════════════════════════════════════════════
def make_objective(arch):
    def objective(trial):
        p = dict(
            W     = trial.suggest_categorical("W", cfg.W_CHOICES),
            L     = trial.suggest_categorical("L", cfg.L_CHOICES),
            beta  = trial.suggest_categorical("beta", cfg.BETA_CHOICES),
            lr    = trial.suggest_float("lr", cfg.LR_LOW, cfg.LR_HIGH, log=True),
            sched = trial.suggest_categorical("sched", cfg.SCHED_CHOICES),
            surr  = trial.suggest_categorical("surr", cfg.SURR_CHOICES),
            fire  = trial.suggest_categorical("fire", cfg.FIRE_CHOICES),
            input = trial.suggest_categorical("input", cfg.INPUT_CHOICES),
        )
        tag = f"{arch}_t{trial.number}_s{cfg.HPO_SEED}"
        print(f"\n[{arch} trial {trial.number}/{cfg.N_TRIALS}] W={p['W']} L={p['L']} beta={p['beta']} "
              f"lr={p['lr']:.1e} {p['sched']} surr={p['surr']} fire={p['fire']} {p['input']}", flush=True)
        r = train_eval(arch, p, cfg.HPO_SEED, tag, trial=trial)
        print(f"   -> trial {trial.number} done: F1={r['f1']:.4f}  acc={r['acc']:.4f}  "
              f"fire={r['firing']:.3f}  @ep{r['best_epoch']}  params={r['params']}", flush=True)
        trial.set_user_attr("params", r["params"])
        trial.set_user_attr("best_epoch", r["best_epoch"])
        trial.set_user_attr("acc", r["acc"])
        return r["f1"], r["firing"]            # (maximize, minimize)
    return objective

def run_study(arch):
    study = optuna.create_study(
        study_name=f"hpo_{arch}", storage=STORAGE, load_if_exists=True,
        directions=["maximize", "minimize"],
        sampler=optuna.samplers.TPESampler(multivariate=True, seed=cfg.HPO_SEED))
    done = len([t for t in study.trials if t.state.name in ("COMPLETE", "PRUNED")])
    todo = max(0, cfg.N_TRIALS - done)
    print(f"\n[STUDY {arch}] {done}/{cfg.N_TRIALS} done, running {todo} more", flush=True)
    if todo:
        study.optimize(make_objective(arch), n_trials=todo, show_progress_bar=False)
    return study

# ══════════════════════════════════════════════════════════════════════════════
# Run all three studies, extract Pareto, select configs, re-evaluate on 3 seeds
# ══════════════════════════════════════════════════════════════════════════════
def select_configs(study):
    """From the Pareto front pick (best_f1) and a firing-efficient (high-F1, min-firing) config."""
    pf = [t for t in study.best_trials]
    if not pf:
        comp = [t for t in study.trials if t.state.name == "COMPLETE"]
        pf = sorted(comp, key=lambda t: t.values[0], reverse=True)[:1]
    best_f1 = max(pf, key=lambda t: t.values[0])
    f1max = best_f1.values[0]
    near = [t for t in pf if t.values[0] >= f1max - 0.01]
    efficient = min(near, key=lambda t: t.values[1])
    out = {"best_f1": best_f1.params}
    if efficient.number != best_f1.number:
        out["efficient"] = efficient.params
    return out, pf

REEVAL_PATH = RUN_DIR / "reeval.json"
reeval = json.loads(REEVAL_PATH.read_text()) if REEVAL_PATH.exists() else {}

studies, pareto = {}, {}
for arch in cfg.ARCHS:
    studies[arch] = run_study(arch)

print("\n" + "═"*78 + "\n PARETO FRONTS  (F1 ↑, firing ↓, params)\n" + "═"*78)
selected = {}
for arch in cfg.ARCHS:
    cfgs, pf = select_configs(studies[arch]); selected[arch] = cfgs; pareto[arch] = []
    print(f"\n[{arch}]  ({len(pf)} Pareto points)")
    for t in sorted(pf, key=lambda x: x.values[0], reverse=True):
        pr = t.user_attrs.get("params", -1)
        pareto[arch].append({"f1": t.values[0], "firing": t.values[1], "params": pr, "params_cfg": t.params})
        print(f"   F1={t.values[0]:.4f}  fire={t.values[1]:.3f}  params={pr:>7}  "
              f"W={t.params['W']} L={t.params['L']} beta={t.params['beta']} lr={t.params['lr']:.1e} "
              f"sched={t.params['sched']} surr={t.params['surr']} fire={t.params['fire']} in={t.params['input']}")

# ── re-evaluate selected configs on 3 seeds ───────────────────────────────────
print("\n" + "═"*78 + "\n RE-EVAL selected configs on seeds " + str(cfg.REEVAL_SEEDS) + "\n" + "═"*78)
for arch in cfg.ARCHS:
    for cname, params in selected[arch].items():
        for seed in cfg.REEVAL_SEEDS:
            key = f"{arch}:{cname}:{seed}"
            if key in reeval:
                continue
            p = dict(W=params["W"], L=params["L"], beta=params["beta"], lr=params["lr"],
                     sched=params["sched"], surr=params["surr"], fire=params["fire"], input=params["input"])
            tag = f"reeval_{arch}_{cname}_s{seed}"
            print(f"\n[re-eval {arch}:{cname} seed {seed}] W={params['W']} L={params['L']} "
                  f"beta={params['beta']} {params['sched']} fire={params['fire']}", flush=True)
            r = train_eval(arch, p, seed, tag)
            reeval[key] = {"f1": r["f1"], "acc": r["acc"], "firing": r["firing"],
                           "params": r["params"], "best_epoch": r["best_epoch"],
                           "preds": r["preds"], "trues": r["trues"]}
            REEVAL_PATH.write_text(json.dumps(reeval))
            print(f"   {key:<34} F1={r['f1']:.4f} acc={r['acc']:.4f} fire={r['firing']:.3f} params={r['params']}", flush=True)

# ── aggregate re-eval + paired bootstrap head-to-head (best_f1 configs) ────────
def agg(arch, cname):
    ks = [f"{arch}:{cname}:{s}" for s in cfg.REEVAL_SEEDS if f"{arch}:{cname}:{s}" in reeval]
    f1s = [reeval[k]["f1"] for k in ks]; firs = [reeval[k]["firing"] for k in ks]
    accs = [reeval[k]["acc"] for k in ks]; pr = reeval[ks[0]]["params"]
    return ks, f1s, firs, accs, pr

print("\n" + "═"*78 + "\n FINAL  (best_f1 config, mean±std over re-eval seeds)\n" + "═"*78)
final = {}
for arch in cfg.ARCHS:
    ks, f1s, firs, accs, pr = agg(arch, "best_f1")
    final[arch] = {"f1_mean": float(np.mean(f1s)), "f1_std": float(np.std(f1s)),
                   "acc_mean": float(np.mean(accs)), "firing_mean": float(np.mean(firs)),
                   "params": pr, "keys": ks}
    print(f"   {arch:<9} F1 {np.mean(f1s):.4f}±{np.std(f1s):.4f}  acc {np.mean(accs):.4f}  "
          f"fire {np.mean(firs):.3f}  params {pr}")

print("\n[BOOTSTRAP] paired on test, per re-eval seed (consistent = same sign all seeds)")
boot = {}
pairs = [("doppler", "lif"), ("doppler", "plif"), ("plif", "lif"), ("d2", "lif"), ("doppler", "d2")]
for a, b in pairs:
    if a not in cfg.ARCHS or b not in cfg.ARCHS: continue
    for label, mfn in [("F1", METRICS["macro_f1"]), ("acc", METRICS["accuracy"])]:
        per_seed = []
        for s in cfg.REEVAL_SEEDS:
            ka, kb = f"{a}:best_f1:{s}", f"{b}:best_f1:{s}"
            if ka in reeval and kb in reeval:
                pa = np.array(reeval[ka]["preds"]); pb = np.array(reeval[kb]["preds"])
                tr = np.array(reeval[ka]["trues"])
                per_seed.append(paired_bootstrap(pa, pb, tr, mfn, C))
        if not per_seed: continue
        mean = float(np.mean([x[0] for x in per_seed]))
        cis = [(round(lo, 3), round(hi, 3)) for _, lo, hi in per_seed]
        consistent = all(lo > 0 for _, lo, _ in per_seed) or all(hi < 0 for _, _, hi in per_seed)
        boot[f"{a}-{b}[{label}]"] = {"mean": mean, "cis": cis, "consistent": consistent}
        print(f"   {a} - {b} [{label}]: mean {mean:+.4f}  CIs {cis}  consistent={consistent}")

# ══════════════════════════════════════════════════════════════════════════════
# CNN baseline on the SAME split  (reference point vs the spiking models)
# ══════════════════════════════════════════════════════════════════════════════
class SmallCNN(nn.Module):
    def __init__(self, n_classes, ch=(16, 32, 64)):
        super().__init__()
        layers, c0 = [], 1
        for c in ch:
            layers += [nn.Conv2d(c0, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(), nn.MaxPool2d(2)]
            c0 = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(ch[-1], n_classes))
    def forward(self, x):                       # x: [N, 64, 64] -> 1-channel image
        return self.head(self.features(x.unsqueeze(1)))

def count_macs_cnn(model):
    macs = [0]
    def ch(m, i, o):
        oh, ow = o.shape[-2], o.shape[-1]; kh, kw = m.kernel_size
        macs[0] += oh*ow*m.out_channels*m.in_channels*kh*kw
    def lh(m, i, o):
        macs[0] += m.in_features*m.out_features
    hs = []
    for mod in model.modules():
        if isinstance(mod, nn.Conv2d): hs.append(mod.register_forward_hook(ch))
        elif isinstance(mod, nn.Linear): hs.append(mod.register_forward_hook(lh))
    model.eval()
    with torch.no_grad(): model(torch.zeros(1, Fin, Fin, device=device))
    for h in hs: h.remove()
    return macs[0]

def cnn_train_eval(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    Xtr, ytr, Xva, yva, Xte, yte = DATA["plain"]
    model = SmallCNN(C).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.MAX_EPOCHS)
    best_val, best, pc, val_hist = -1.0, None, 0, []
    use_bar = cfg.USE_TQDM and _HAS_TQDM
    it = tqdm(range(cfg.MAX_EPOCHS), desc=f"cnn_s{seed}", leave=False) if use_bar else range(cfg.MAX_EPOCHS)
    for ep in it:
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), cfg.BATCH):
            b = perm[i:i+cfg.BATCH]; opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(Xtr[b].to(device)), ytr[b].to(device)).backward(); opt.step()
        sched.step()
        vp, vt = evaluate(model, Xva, yva, cfg.BATCH); v = macro_f1(vp, vt, C)[0]; val_hist.append(v)
        vs = float(np.mean(val_hist[-cfg.VAL_SMOOTH:]))
        tp, tt = evaluate(model, Xte, yte, cfg.BATCH); tf1, _ = macro_f1(tp, tt, C); ta = accuracy(tp, tt)
        if use_bar: it.set_postfix(val=f"{vs:.3f}", f1=f"{tf1:.3f}", pat=pc)
        if vs > best_val + 1e-4:
            best_val, pc = vs, 0
            best = {"f1": tf1, "acc": ta, "preds": tp.tolist(), "trues": tt.tolist()}
        else:
            pc += 1
        if pc >= cfg.PATIENCE: break
    if use_bar: it.close()
    best["params"] = count_params(model); best["macs"] = count_macs_cnn(model)
    return best

print("\n" + "═"*78 + "\n CNN BASELINE on the same split (seeds " + str(cfg.REEVAL_SEEDS) + ")\n" + "═"*78)
CNN_PATH = RUN_DIR / "cnn.json"
cnn = json.loads(CNN_PATH.read_text()) if CNN_PATH.exists() else {}
for s in cfg.REEVAL_SEEDS:
    k = f"cnn:{s}"
    if k in cnn: continue
    print(f"\n[cnn seed {s}]", flush=True)
    r = cnn_train_eval(s); cnn[k] = r; CNN_PATH.write_text(json.dumps(cnn))
    print(f"   cnn:{s}  F1={r['f1']:.4f} acc={r['acc']:.4f} params={r['params']} macs={r['macs']}", flush=True)
cnn_f1 = [cnn[f"cnn:{s}"]["f1"] for s in cfg.REEVAL_SEEDS if f"cnn:{s}" in cnn]
cnn_acc = [cnn[f"cnn:{s}"]["acc"] for s in cfg.REEVAL_SEEDS if f"cnn:{s}" in cnn]
cnn_params = cnn[f"cnn:{cfg.REEVAL_SEEDS[0]}"]["params"]; cnn_macs = cnn[f"cnn:{cfg.REEVAL_SEEDS[0]}"]["macs"]
print(f"\n   CNN  F1 {np.mean(cnn_f1):.4f}±{np.std(cnn_f1):.4f}  acc {np.mean(cnn_acc):.4f}  "
      f"params {cnn_params}  macs {cnn_macs/1e6:.2f}M", flush=True)

print("\n[BOOTSTRAP vs CNN] (informational; CNN is a different, dense model class)")
for a in [x for x in cfg.ARCHS if x in ("doppler", "lif", "plif")]:
    for label, mfn in [("F1", METRICS["macro_f1"]), ("acc", METRICS["accuracy"])]:
        per_seed = []
        for s in cfg.REEVAL_SEEDS:
            ka, kc = f"{a}:best_f1:{s}", f"cnn:{s}"
            if ka in reeval and kc in cnn:
                pa = np.array(reeval[ka]["preds"]); pc_ = np.array(cnn[kc]["preds"]); tr = np.array(reeval[ka]["trues"])
                per_seed.append(paired_bootstrap(pa, pc_, tr, mfn, C))
        if not per_seed: continue
        mean = float(np.mean([x[0] for x in per_seed]))
        cis = [(round(lo, 3), round(hi, 3)) for _, lo, hi in per_seed]
        consistent = all(lo > 0 for _, lo, _ in per_seed) or all(hi < 0 for _, _, hi in per_seed)
        boot[f"{a}-cnn[{label}]"] = {"mean": mean, "cis": cis, "consistent": consistent}
        print(f"   {a} - cnn [{label}]: mean {mean:+.4f}  CIs {cis}  consistent={consistent}")

# ══════════════════════════════════════════════════════════════════════════════
# Energy proxy  (SynOps + estimated 45nm energy/inference; rough, documented model)
# ══════════════════════════════════════════════════════════════════════════════
# T=64 timesteps (one per Doppler-time slice). Layer-1 = dense MAC (analog input);
# layers 2..L = spike-driven accumulates (AC) gated by mean firing. 45nm Horowitz:
# E_MAC=4.6 pJ (32b mult-add), E_AC=0.9 pJ (32b add). Ignores memory movement and
# input encoding; this is a RELATIVE comparison, not absolute silicon energy.
E_MAC, E_AC, T = 4.6e-12, 0.9e-12, Fin
def snn_energy(W, L, firing):
    macs = Fin*W*T + W*C                        # layer-1 dense + readout
    syn  = (L-1)*(firing*W*W*T)                 # spike-driven hidden layers
    return macs, syn, macs*E_MAC + syn*E_AC

print("\n" + "═"*78 + "\n ENERGY PROXY  (per inference; 45nm estimate, relative only)\n" + "═"*78)
energy = {}
for a in [x for x in cfg.ARCHS if x in ("doppler", "lif", "plif")]:
    ca = selected[a]["best_f1"]; W, L = ca["W"], ca["L"]; fr = final[a]["firing_mean"]
    macs, syn, E = snn_energy(W, L, fr)
    energy[a] = {"macs": macs, "synops": syn, "nJ": E*1e9, "W": W, "L": L, "firing": fr}
    print(f"   {a:<9} W={W} L={L} fire={fr:.3f}  MAC={macs/1e6:.2f}M  SynOps={syn/1e6:.2f}M  ~{E*1e9:.1f} nJ")
cnn_E = cnn_macs*E_MAC
energy["cnn"] = {"macs": cnn_macs, "synops": 0, "nJ": cnn_E*1e9}
print(f"   {'cnn':<9} (dense)        MAC={cnn_macs/1e6:.2f}M  SynOps=0.00M       ~{cnn_E*1e9:.1f} nJ")
if "doppler" in energy:
    print(f"\n   energy ratio  CNN / Doppler = {cnn_E/(energy['doppler']['nJ']*1e-9):.1f}x  "
          f"(Doppler {energy['doppler']['nJ']:.1f} nJ vs CNN {cnn_E*1e9:.1f} nJ)")

# ── save ──────────────────────────────────────────────────────────────────────
summary = {"run_dir": str(RUN_DIR), "archs": cfg.ARCHS, "n_trials": cfg.N_TRIALS,
           "reeval_seeds": cfg.REEVAL_SEEDS, "search_space": {
               "W": cfg.W_CHOICES, "L": cfg.L_CHOICES, "beta": cfg.BETA_CHOICES,
               "lr": [cfg.LR_LOW, cfg.LR_HIGH], "sched": cfg.SCHED_CHOICES, "surr": cfg.SURR_CHOICES,
               "fire": cfg.FIRE_CHOICES, "input": cfg.INPUT_CHOICES},
           "selected": selected, "pareto": pareto, "final": final, "bootstrap": boot,
           "cnn": {"f1_mean": float(np.mean(cnn_f1)), "f1_std": float(np.std(cnn_f1)),
                   "acc_mean": float(np.mean(cnn_acc)), "params": cnn_params, "macs": cnn_macs},
           "energy": energy,
           "timestamp": datetime.now().isoformat()}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"\n[DONE] summary -> {RUN_DIR / 'summary.json'}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CROSS-DOMAIN DISSOCIATION  ·  SHD (audio) per-neuron homogeneous bake-off
# Counterpart to the DIAT-µSAT ablation. Tests whether the neuron RANKING changes
# across domains: on DIAT (radar, frequency-structured) Doppler-LIF won; the
# pre-registered prediction is that on SHD (audio, temporal) Doppler-LIF is NOT
# the top neuron and a temporally-matched neuron (primary: Dual-tau-LIF) wins.
# Double dissociation holds iff  SHD-winner ≠ Doppler  while  DIAT-winner = Doppler.
#
# Controlled comparison (NOT HPO): identical config across neurons, matched to the
# DIAT ablation (L=4, W=128, target_firing=0.15, val-smoothed macro-F1, 3 seeds).
# Selection machinery identical to DIAT (init calibration, smoothed-val, test once).
#
# NOTE: this SHD cache is speaker-MIXED (not the official speaker-disjoint split),
# so absolute accuracy is NOT comparable to SHD literature. The dissociation only
# uses within-split neuron RANKING, which the split choice does not affect.
#
# Single paste-and-run Colab cell. SQLite-free; checkpoint + results.json resume.
# ══════════════════════════════════════════════════════════════════════════════
import os, json, math, copy, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ────────────────────────────── CONFIG ────────────────────────────────────────
class CFG:
    # candidates: full registry + vanilla baseline. DIAT-overlap set (direct crossover)
    # = {Doppler-LIF, Chirp-LIF, STFT-IF, Dual-tau-LIF}.
    NEURONS = ["Doppler-LIF", "Dual-tau-LIF", "Phase-LIF", "CrossInhib-LIF",
               "Chirp-LIF", "Beam-IF", "STFT-IF", "Vanilla-LIF"]
    L            = 4
    W            = 128
    TARGET_FIRING = 0.15
    NORM         = "none"          # pure SNN
    MAX_EPOCHS   = 200
    PATIENCE     = 20
    VAL_SMOOTH   = 5
    VAL_FRAC     = 0.15
    LR           = 1e-3
    BATCH        = 128
    SELECT_METRIC = "macro_f1"
    SEEDS        = [42, 123, 999]
    USE_TQDM     = True
    HEARTBEAT_EVERY = 5
    if IN_COLAB:
        PROJECT = "/content/drive/MyDrive/Research/NISAC_DeepHetero"
    else:
        PROJECT = "/home/claude/shd_run"

cfg = CFG()

if os.environ.get("REDUCED") == "1":
    cfg.NEURONS = ["Doppler-LIF", "Dual-tau-LIF", "Vanilla-LIF"]
    cfg.L = 2; cfg.W = 24; cfg.MAX_EPOCHS = 6; cfg.PATIENCE = 4; cfg.VAL_SMOOTH = 2
    cfg.SEEDS = [42]

# DIAT homogeneous ranking (locked ablation, macro-F1) for the side-by-side crossover
DIAT_HOMOG = {"Doppler-LIF": 0.8719, "Dual-tau-LIF": 0.8598,
              "Chirp-LIF": 0.6971, "STFT-IF": 0.1850}

if IN_COLAB:
    os.system("pip install -q git+https://github.com/DrCanD/dikmen-spiking-neurons.git tqdm")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import dikmen_neurons as D
from dikmen_neurons import BaseNeuron, spike_hard, NeuronRegistry
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ID  = os.environ.get("RUN_ID", "shd_dissoc")
RUN_DIR = Path(cfg.PROJECT) / "dissoc" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = RUN_DIR / "ckpt"; CKPT_DIR.mkdir(exist_ok=True)
print(f"[HW] device={device}" + (f"  gpu={torch.cuda.get_device_name(0)}" if device=='cuda' else ""), flush=True)
print(f"[RUN] {RUN_DIR}", flush=True)

# ── matched Vanilla-LIF baseline (same surrogate/interface, plain dynamics) ─────
class VanillaLIF(BaseNeuron):
    _family = "lif"
    def single_step(self, x_t, state):
        mem = self.beta * state["mem"] + x_t
        spk = spike_hard(mem, self.threshold)
        return spk, {"mem": mem * (1.0 - spk)}
NeuronRegistry._all["Vanilla-LIF"] = VanillaLIF

# ══════════════════════════════════════════════════════════════════════════════
# SHD loader  (verified; .pt dict [N,T=12,F=700], 20 classes, speaker-mixed cache)
# ══════════════════════════════════════════════════════════════════════════════
REGISTRY_PATH = Path(cfg.PROJECT) / "datasets.json"
SHD_PATH = "/content/drive/MyDrive/Research/GHN_Instinct/gesture_cache/shd_T12_m200_v2.pt"

def load_shd():
    if not IN_COLAB:                      # sandbox: synthetic SHD-shaped data
        g = torch.Generator().manual_seed(0)
        N, T, Fin, C = 240, 12, 700, 20
        X = torch.rand(N, T, Fin, generator=g)
        y = torch.randint(0, C, (N,), generator=g)
        return X[:190], y[:190], X[190:], y[190:]
    obj = torch.load(SHD_PATH, map_location="cpu", weights_only=False)
    if isinstance(obj, dict) and "Xtr" in obj:
        Xtr, ytr, Xte, yte = obj["Xtr"], obj["ytr"], obj["Xte"], obj["yte"]
    elif isinstance(obj, dict) and "X" in obj:
        X, y = obj["X"], obj["y"]
        n = len(X); idx = torch.randperm(n, generator=torch.Generator().manual_seed(0)); cut = int(0.8*n)
        Xtr, ytr, Xte, yte = X[idx[:cut]], y[idx[:cut]], X[idx[cut:]], y[idx[cut:]]
    else:
        raise ValueError(f"Unexpected SHD keys: {list(obj.keys())}")
    to_t = lambda a: torch.as_tensor(np.asarray(a)).float()
    to_y = lambda a: torch.as_tensor(np.asarray(a)).long()
    Xtr, Xte, ytr, yte = to_t(Xtr), to_t(Xte), to_y(ytr), to_y(yte)
    assert Xtr.ndim == 3, f"expected [N,T,F], got {tuple(Xtr.shape)}"
    return Xtr, ytr, Xte, yte

def _strat_val(Xa, ya, frac, seed):
    try:
        from sklearn.model_selection import train_test_split
        A, B, ya2, yb2 = train_test_split(Xa.numpy(), ya.numpy(), test_size=frac,
                                           stratify=ya.numpy(), random_state=seed)
        return (torch.as_tensor(A).float(), torch.as_tensor(ya2).long(),
                torch.as_tensor(B).float(), torch.as_tensor(yb2).long())
    except Exception:
        rng = np.random.default_rng(seed); ia, ib = [], []
        for c in torch.unique(ya).tolist():
            ci = np.where(ya.numpy() == c)[0]; rng.shuffle(ci); k = int((1-frac)*len(ci))
            ia += list(ci[:k]); ib += list(ci[k:])
        return Xa[ia], ya[ia], Xa[ib], ya[ib]

Xtrall, ytrall, Xte, yte = load_shd()
# standardize with train stats (global), same as DIAT pipeline
mu, sd = float(Xtrall.mean()), float(Xtrall.std() + 1e-6)
Xtrall = (Xtrall - mu)/sd; Xte = (Xte - mu)/sd
Xtr, ytr, Xva, yva = _strat_val(Xtrall, ytrall, cfg.VAL_FRAC, 0)
C = int(max(ytr.max(), yte.max()) + 1); Fin = Xtr.shape[-1]; Tlen = Xtr.shape[1]
print(f"[DATA] SHD train={tuple(Xtr.shape)} val={tuple(Xva.shape)} test={tuple(Xte.shape)} "
      f"T={Tlen} F={Fin} C={C} chance={1/C:.3f}", flush=True)

# ══════════════════════════════════════════════════════════════════════════════
# Harness (inline) — identical to the verified DIAT cells
# ══════════════════════════════════════════════════════════════════════════════
def _make_norm(kind, width):
    if kind == "layer": return nn.LayerNorm(width)
    if kind == "batch": return nn.BatchNorm1d(width)
    return nn.Identity()

class FeatureStack(nn.Module):
    def __init__(self, in_features, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        neuron_kwargs = neuron_kwargs or {}
        dims = [in_features] + [width]*len(layer_types)
        self.projs = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(layer_types))])
        self.norms = nn.ModuleList([_make_norm(norm, width) for _ in layer_types])
        self.neurons = nn.ModuleList(
            [NeuronRegistry.create(t, size=width, **neuron_kwargs.get(t, {})) for t in layer_types])
    def forward(self, x):
        h = x
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); h = spk
        return h.mean(dim=1), h
    def layer_firing(self, x):
        h, rates = x, []
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); rates.append(spk.float().mean().item()); h = spk
        return rates

class VerticalNet(nn.Module):
    def __init__(self, in_features, n_classes, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        self.stack = FeatureStack(in_features, layer_types, width, neuron_kwargs, norm)
        self.readout = nn.Linear(width, n_classes)
    def forward(self, x):
        feat, _ = self.stack(x); return self.readout(feat)
    def firing_rates(self, x): return self.stack.layer_firing(x)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def calibrate_thresholds(model, x, target, iters=30):
    h = x
    for proj, norm, neuron in zip(model.stack.projs, model.stack.norms, model.stack.neurons):
        B, T, d = h.shape
        z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
        a, b = 1e-3, 1e5
        for _ in range(iters):
            mid = (a*b)**0.5; neuron.threshold = mid
            r = neuron(z)[0].float().mean().item()
            if r > target: a = mid
            else:          b = mid
        neuron.threshold = (a*b)**0.5
        h = neuron(z)[0]
    return model

def accuracy(p, t): return float((p == t).mean())
def macro_f1(p, t, C):
    f1s = []
    for c in range(C):
        tp = int(np.sum((p==c)&(t==c))); fp = int(np.sum((p==c)&(t!=c))); fn = int(np.sum((p!=c)&(t==c)))
        pr = tp/(tp+fp) if (tp+fp) else 0.0; rc = tp/(tp+fn) if (tp+fn) else 0.0
        f1s.append(2*pr*rc/(pr+rc) if (pr+rc) else 0.0)
    return float(np.mean(f1s))
METRICS = {"macro_f1": lambda p,t,C: macro_f1(p,t,C), "accuracy": lambda p,t,C: accuracy(p,t)}

@torch.no_grad()
def evaluate(model, X, Y, batch):
    model.eval(); preds = []
    for i in range(0, len(X), batch):
        preds.append(model(X[i:i+batch].to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy(), Y.numpy()

def firing_mean(model, xb): return float(np.mean(model.firing_rates(xb)))

def paired_bootstrap(pa, pb, trues, mfn, C, nboot=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(trues); boots = np.empty(nboot)
    base = mfn(pa, trues, C) - mfn(pb, trues, C)
    for k in range(nboot):
        i = rng.integers(0, n, n)
        boots[k] = mfn(pa[i], trues[i], C) - mfn(pb[i], trues[i], C)
    return float(base), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

# ══════════════════════════════════════════════════════════════════════════════
# Train one homogeneous net (val-smoothed selection, test once at best epoch)
# ══════════════════════════════════════════════════════════════════════════════
def train_one(neuron, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    fire_batch = Xtr[:256].to(device)
    model = VerticalNet(Fin, C, [neuron]*cfg.L, cfg.W, {}, cfg.NORM).to(device)
    calibrate_thresholds(model, fire_batch, cfg.TARGET_FIRING)
    n_params = count_params(model)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.LR)
    sel = METRICS[cfg.SELECT_METRIC]

    ckpt = CKPT_DIR / f"{neuron.replace('/','_')}_s{seed}.pt"
    start, m = 0, {}
    if ckpt.exists():
        ck = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); start, m = ck["epoch"]+1, ck["m"]
    val_hist = m.get("val_hist", []); best_val = m.get("best_val", -1.0)
    best_epoch = m.get("best_epoch", -1); pc = m.get("pc", 0); best = m.get("best", None)

    use_bar = cfg.USE_TQDM and _HAS_TQDM
    it = tqdm(range(start, cfg.MAX_EPOCHS), desc=f"{neuron} s{seed}", leave=False,
              initial=start, total=cfg.MAX_EPOCHS) if use_bar else range(start, cfg.MAX_EPOCHS)
    for ep in it:
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), cfg.BATCH):
            b = perm[i:i+cfg.BATCH]
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(Xtr[b].to(device)), ytr[b].to(device)).backward(); opt.step()
        vp, vt = evaluate(model, Xva, yva, cfg.BATCH); val_hist.append(sel(vp, vt, C))
        v_smooth = float(np.mean(val_hist[-cfg.VAL_SMOOTH:]))
        tp, tt = evaluate(model, Xte, yte, cfg.BATCH)
        t_f1 = macro_f1(tp, tt, C); t_acc = accuracy(tp, tt)
        improved = v_smooth > best_val + 1e-4
        if improved:
            best_val, best_epoch, pc = v_smooth, ep, 0
            best = {"f1": t_f1, "acc": t_acc, "firing": firing_mean(model, fire_batch),
                    "preds": tp.tolist(), "trues": tt.tolist()}
        else:
            pc += 1
        torch.save({"epoch": ep, "model": copy.deepcopy(model.state_dict()), "opt": opt.state_dict(),
                    "m": {"val_hist": val_hist, "best_val": best_val, "best_epoch": best_epoch,
                          "pc": pc, "best": best}}, ckpt)
        cur = best["firing"] if best else 0.0
        if use_bar:
            it.set_postfix(val=f"{v_smooth:.3f}", f1=f"{t_f1:.3f}", fire=f"{cur:.2f}", pat=pc)
        elif (ep % cfg.HEARTBEAT_EVERY == 0) or improved:
            print(f"      ep{ep:>3}/{cfg.MAX_EPOCHS} val={v_smooth:.4f} f1={t_f1:.4f} "
                  f"fire={cur:.3f} pat={pc}{'  *' if improved else ''}", flush=True)
        if pc >= cfg.PATIENCE: break
    if use_bar: it.close()
    best["params"] = n_params; best["best_epoch"] = best_epoch
    return best

# ══════════════════════════════════════════════════════════════════════════════
# Run all neurons × seeds  (resume via results.json)
# ══════════════════════════════════════════════════════════════════════════════
RES_PATH = RUN_DIR / "results.json"
res = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
print("\n" + "═"*78 + "\n SHD per-neuron bake-off (homogeneous, matched config)\n" + "═"*78, flush=True)
for neuron in cfg.NEURONS:
    for seed in cfg.SEEDS:
        key = f"{neuron}:{seed}"
        if key in res: continue
        print(f"\n[{neuron} seed {seed}]", flush=True)
        r = train_one(neuron, seed)
        res[key] = r; RES_PATH.write_text(json.dumps(res))
        print(f"   -> {key}: F1={r['f1']:.4f} acc={r['acc']:.4f} fire={r['firing']:.3f} "
              f"@ep{r['best_epoch']} params={r['params']}", flush=True)

# ══════════════════════════════════════════════════════════════════════════════
# Aggregate ranking + dissociation crossover + bootstrap
# ══════════════════════════════════════════════════════════════════════════════
def agg(neuron):
    ks = [f"{neuron}:{s}" for s in cfg.SEEDS if f"{neuron}:{s}" in res]
    f1 = [res[k]["f1"] for k in ks]; ac = [res[k]["acc"] for k in ks]; fr = [res[k]["firing"] for k in ks]
    return ks, f1, ac, fr

print("\n" + "═"*78 + "\n SHD RANKING  (macro-F1, mean±std over seeds)\n" + "═"*78, flush=True)
ranking = []
for neuron in cfg.NEURONS:
    ks, f1, ac, fr = agg(neuron)
    if not ks: continue
    ranking.append((neuron, float(np.mean(f1)), float(np.std(f1)), float(np.mean(ac)), float(np.mean(fr))))
ranking.sort(key=lambda x: x[1], reverse=True)
for i,(n,f,s,a,fr) in enumerate(ranking):
    tag = "  <-- DIAT winner" if n == "Doppler-LIF" else ""
    print(f"   {i+1}. {n:16s} F1 {f:.4f}±{s:.4f}  acc {a:.4f}  fire {fr:.3f}{tag}", flush=True)

shd_winner = ranking[0][0] if ranking else None
print("\n" + "═"*78 + "\n DOUBLE DISSOCIATION  (DIAT-overlap neurons)\n" + "═"*78, flush=True)
print(f"   {'neuron':16s}   DIAT-F1    SHD-F1", flush=True)
for n in ["Doppler-LIF", "Dual-tau-LIF", "Chirp-LIF", "STFT-IF"]:
    shd = next((r[1] for r in ranking if r[0]==n), None)
    d = DIAT_HOMOG.get(n)
    print(f"   {n:16s}   {d:.4f}    {shd:.4f}" if shd is not None else f"   {n:16s}   {d:.4f}    (n/a)", flush=True)
print(f"\n   DIAT winner = Doppler-LIF   |   SHD winner = {shd_winner}", flush=True)
dissociation = (shd_winner is not None and shd_winner != "Doppler-LIF")
print(f"   DISSOCIATION {'CONFIRMED' if dissociation else 'NOT confirmed'} "
      f"(SHD winner {'≠' if dissociation else '=='} Doppler)", flush=True)

print("\n[BOOTSTRAP] paired on SHD test, per seed (consistent = same sign all seeds)", flush=True)
boot = {}
pairs = []
if shd_winner and shd_winner != "Doppler-LIF": pairs.append((shd_winner, "Doppler-LIF"))
pairs += [("Doppler-LIF", "Vanilla-LIF"), ("Dual-tau-LIF", "Doppler-LIF")]
seen = set()
for a, b in pairs:
    if (a,b) in seen or a == b: continue
    seen.add((a,b))
    if not (any(f"{a}:{s}" in res for s in cfg.SEEDS) and any(f"{b}:{s}" in res for s in cfg.SEEDS)): continue
    for label, mfn in [("F1", METRICS["macro_f1"]), ("acc", METRICS["accuracy"])]:
        per = []
        for s in cfg.SEEDS:
            ka, kb = f"{a}:{s}", f"{b}:{s}"
            if ka in res and kb in res:
                per.append(paired_bootstrap(np.array(res[ka]["preds"]), np.array(res[kb]["preds"]),
                                            np.array(res[ka]["trues"]), mfn, C))
        if not per: continue
        mean = float(np.mean([x[0] for x in per])); cis = [(round(l,3), round(h,3)) for _,l,h in per]
        consistent = all(l>0 for _,l,_ in per) or all(h<0 for _,_,h in per)
        boot[f"{a}-{b}[{label}]"] = {"mean": mean, "cis": cis, "consistent": consistent}
        print(f"   {a} - {b} [{label}]: mean {mean:+.4f}  CIs {cis}  consistent={consistent}", flush=True)

summary = {"run_dir": str(RUN_DIR), "config": {"L": cfg.L, "W": cfg.W, "target_firing": cfg.TARGET_FIRING,
           "seeds": cfg.SEEDS, "max_epochs": cfg.MAX_EPOCHS}, "ranking": ranking,
           "shd_winner": shd_winner, "dissociation": dissociation, "diat_homog": DIAT_HOMOG,
           "bootstrap": boot, "timestamp": datetime.now().isoformat()}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"\n[DONE] summary -> {RUN_DIR / 'summary.json'}", flush=True)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CLOSE THE 2×2  ·  DIAT-µSAT (radar) homogeneous bake-off for the cross neurons
# Completes the double-dissociation square. SHD (audio) winner was CrossInhib-LIF
# (Doppler 6th of 8). This runs {Doppler, CrossInhib, Phase} homogeneously on DIAT
# under the EXACT ablation config to show the mirror image: Doppler wins DIAT and
# CrossInhib falls below it. Doppler is re-run both for the paired bootstrap (same
# test split) and as a harness-consistency check (must reproduce ~0.872).
#
# 2×2 double dissociation CONFIRMED iff:
#     Doppler_DIAT > CrossInhib_DIAT   AND   CrossInhib_SHD > Doppler_SHD
#
# Config identical to the DIAT ablation and the SHD cell (L=4, W=128, fire=0.15,
# val-smoothed macro-F1, 3 seeds). Single paste-and-run Colab cell, resumable.
# ══════════════════════════════════════════════════════════════════════════════
import os, json, math, copy, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ────────────────────────────── CONFIG ────────────────────────────────────────
class CFG:
    NEURONS = ["Doppler-LIF", "CrossInhib-LIF", "Phase-LIF"]   # Doppler re-run for paired bootstrap
    L            = 4
    W            = 128
    TARGET_FIRING = 0.15
    NORM         = "none"
    MAX_EPOCHS   = 200
    PATIENCE     = 20
    VAL_SMOOTH   = 5
    VAL_FRAC     = 0.15
    LR           = 1e-3
    BATCH        = 128
    SELECT_METRIC = "macro_f1"
    SEEDS        = [42, 123, 999]
    USE_TQDM     = True
    HEARTBEAT_EVERY = 5
    PROJECT = "/content/drive/MyDrive/Research/NISAC_DeepHetero" if IN_COLAB else "/home/claude/diat_run"

cfg = CFG()
if os.environ.get("REDUCED") == "1":
    cfg.NEURONS = ["Doppler-LIF", "CrossInhib-LIF", "Phase-LIF"]
    cfg.L = 2; cfg.W = 24; cfg.MAX_EPOCHS = 6; cfg.PATIENCE = 4; cfg.VAL_SMOOTH = 2
    cfg.SEEDS = [42]

# locked DIAT ablation homogeneous (macro-F1) — Doppler value is the reproduction target
DIAT_HOMOG = {"Doppler-LIF": 0.8719, "Dual-tau-LIF": 0.8598, "Chirp-LIF": 0.6971, "STFT-IF": 0.1850}
# measured SHD homogeneous ranking (this study) — the cross-domain reference
SHD_HOMOG = {"CrossInhib-LIF": 0.7405, "Vanilla-LIF": 0.7030, "Dual-tau-LIF": 0.6943,
             "Chirp-LIF": 0.6624, "Beam-IF": 0.6385, "Doppler-LIF": 0.6005,
             "STFT-IF": 0.2439, "Phase-LIF": 0.1430}

if IN_COLAB:
    os.system("pip install -q git+https://github.com/DrCanD/dikmen-spiking-neurons.git tqdm")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import dikmen_neurons as D
from dikmen_neurons import BaseNeuron, spike_hard, NeuronRegistry
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ID  = os.environ.get("RUN_ID", "diat_2x2")
RUN_DIR = Path(cfg.PROJECT) / "dissoc" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = RUN_DIR / "ckpt"; CKPT_DIR.mkdir(exist_ok=True)
print(f"[HW] device={device}" + (f"  gpu={torch.cuda.get_device_name(0)}" if device=='cuda' else ""), flush=True)
print(f"[RUN] {RUN_DIR}", flush=True)

# Vanilla baseline registered for parity (not in NEURONS by default, but harmless)
class VanillaLIF(BaseNeuron):
    _family = "lif"
    def single_step(self, x_t, state):
        mem = self.beta * state["mem"] + x_t
        spk = spike_hard(mem, self.threshold)
        return spk, {"mem": mem * (1.0 - spk)}
NeuronRegistry._all["Vanilla-LIF"] = VanillaLIF

# ══════════════════════════════════════════════════════════════════════════════
# DIAT loader (verbatim from the verified ablation: [N,64,64]=[N,time,Doppler],
# 80/20 stratified seed=0, val carved from trainval, train-only standardization)
# ══════════════════════════════════════════════════════════════════════════════
DIAT_PATH = "/content/drive/MyDrive/NISAC/data/DIAT_uSAT/processed"

def _diat_xy():
    if not IN_COLAB:                       # sandbox: synthetic DIAT-shaped data
        g = np.random.default_rng(0)
        return g.standard_normal((240, 64, 64)).astype(np.float32), g.integers(0, 6, 240).astype(np.int64)
    X = np.load(Path(DIAT_PATH) / "X.npy").astype(np.float32)   # [N,64,64]
    y = np.load(Path(DIAT_PATH) / "y.npy").astype(np.int64)
    return X, y

def load_diat():
    X, y = _diat_xy()
    assert X.ndim == 3, f"expected [N,64,64], got {X.shape}"
    def strat_split(Xa, ya, frac, seed):
        try:
            from sklearn.model_selection import train_test_split
            A, B, ya2, yb2 = train_test_split(Xa, ya, test_size=frac, stratify=ya, random_state=seed)
            return A, ya2, B, yb2
        except Exception:
            rng = np.random.default_rng(seed); ia, ib = [], []
            for c in np.unique(ya):
                ci = np.where(ya == c)[0]; rng.shuffle(ci); k = int((1 - frac) * len(ci))
                ia += list(ci[:k]); ib += list(ci[k:])
            return Xa[ia], ya[ia], Xa[ib], ya[ib]
    Xtv, ytv, Xte, yte = strat_split(X, y, 0.20, 0)
    Xtr, ytr, Xva, yva = strat_split(Xtv, ytv, cfg.VAL_FRAC / 0.80, 0)
    mu, sd = float(Xtr.mean()), float(Xtr.std() + 1e-6)
    norm = lambda a: torch.as_tensor((a - mu) / sd).float()
    yy   = lambda a: torch.as_tensor(a).long()
    return norm(Xtr), yy(ytr), norm(Xva), yy(yva), norm(Xte), yy(yte)

Xtr, ytr, Xva, yva, Xte, yte = load_diat()
C = int(max(ytr.max(), yte.max()) + 1); Fin = Xtr.shape[-1]; Tlen = Xtr.shape[1]
print(f"[DATA] DIAT train={tuple(Xtr.shape)} val={tuple(Xva.shape)} test={tuple(Xte.shape)} "
      f"T={Tlen} F={Fin} C={C} chance={1/C:.3f}", flush=True)

# ══════════════════════════════════════════════════════════════════════════════
# Harness (inline) — identical to the verified DIAT/SHD cells
# ══════════════════════════════════════════════════════════════════════════════
def _make_norm(kind, width):
    if kind == "layer": return nn.LayerNorm(width)
    if kind == "batch": return nn.BatchNorm1d(width)
    return nn.Identity()

class FeatureStack(nn.Module):
    def __init__(self, in_features, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        neuron_kwargs = neuron_kwargs or {}
        dims = [in_features] + [width]*len(layer_types)
        self.projs = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(layer_types))])
        self.norms = nn.ModuleList([_make_norm(norm, width) for _ in layer_types])
        self.neurons = nn.ModuleList(
            [NeuronRegistry.create(t, size=width, **neuron_kwargs.get(t, {})) for t in layer_types])
    def forward(self, x):
        h = x
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); h = spk
        return h.mean(dim=1), h
    def layer_firing(self, x):
        h, rates = x, []
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); rates.append(spk.float().mean().item()); h = spk
        return rates

class VerticalNet(nn.Module):
    def __init__(self, in_features, n_classes, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        self.stack = FeatureStack(in_features, layer_types, width, neuron_kwargs, norm)
        self.readout = nn.Linear(width, n_classes)
    def forward(self, x):
        feat, _ = self.stack(x); return self.readout(feat)
    def firing_rates(self, x): return self.stack.layer_firing(x)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def calibrate_thresholds(model, x, target, iters=30):
    h = x
    for proj, norm, neuron in zip(model.stack.projs, model.stack.norms, model.stack.neurons):
        B, T, d = h.shape
        z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
        a, b = 1e-3, 1e5
        for _ in range(iters):
            mid = (a*b)**0.5; neuron.threshold = mid
            r = neuron(z)[0].float().mean().item()
            if r > target: a = mid
            else:          b = mid
        neuron.threshold = (a*b)**0.5
        h = neuron(z)[0]
    return model

def accuracy(p, t): return float((p == t).mean())
def macro_f1(p, t, C):
    f1s = []
    for c in range(C):
        tp = int(np.sum((p==c)&(t==c))); fp = int(np.sum((p==c)&(t!=c))); fn = int(np.sum((p!=c)&(t==c)))
        pr = tp/(tp+fp) if (tp+fp) else 0.0; rc = tp/(tp+fn) if (tp+fn) else 0.0
        f1s.append(2*pr*rc/(pr+rc) if (pr+rc) else 0.0)
    return float(np.mean(f1s))
METRICS = {"macro_f1": lambda p,t,C: macro_f1(p,t,C), "accuracy": lambda p,t,C: accuracy(p,t)}

@torch.no_grad()
def evaluate(model, X, Y, batch):
    model.eval(); preds = []
    for i in range(0, len(X), batch):
        preds.append(model(X[i:i+batch].to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy(), Y.numpy()

def firing_mean(model, xb): return float(np.mean(model.firing_rates(xb)))

def paired_bootstrap(pa, pb, trues, mfn, C, nboot=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(trues); boots = np.empty(nboot)
    base = mfn(pa, trues, C) - mfn(pb, trues, C)
    for k in range(nboot):
        i = rng.integers(0, n, n)
        boots[k] = mfn(pa[i], trues[i], C) - mfn(pb[i], trues[i], C)
    return float(base), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

# ══════════════════════════════════════════════════════════════════════════════
# Train one homogeneous net (val-smoothed selection, test once at best epoch)
# ══════════════════════════════════════════════════════════════════════════════
def train_one(neuron, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    fire_batch = Xtr[:256].to(device)
    model = VerticalNet(Fin, C, [neuron]*cfg.L, cfg.W, {}, cfg.NORM).to(device)
    calibrate_thresholds(model, fire_batch, cfg.TARGET_FIRING)
    n_params = count_params(model)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.LR)
    sel = METRICS[cfg.SELECT_METRIC]

    ckpt = CKPT_DIR / f"{neuron.replace('/','_')}_s{seed}.pt"
    start, m = 0, {}
    if ckpt.exists():
        ck = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); start, m = ck["epoch"]+1, ck["m"]
    val_hist = m.get("val_hist", []); best_val = m.get("best_val", -1.0)
    best_epoch = m.get("best_epoch", -1); pc = m.get("pc", 0); best = m.get("best", None)

    use_bar = cfg.USE_TQDM and _HAS_TQDM
    it = tqdm(range(start, cfg.MAX_EPOCHS), desc=f"{neuron} s{seed}", leave=False,
              initial=start, total=cfg.MAX_EPOCHS) if use_bar else range(start, cfg.MAX_EPOCHS)
    for ep in it:
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), cfg.BATCH):
            b = perm[i:i+cfg.BATCH]
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(Xtr[b].to(device)), ytr[b].to(device)).backward(); opt.step()
        vp, vt = evaluate(model, Xva, yva, cfg.BATCH); val_hist.append(sel(vp, vt, C))
        v_smooth = float(np.mean(val_hist[-cfg.VAL_SMOOTH:]))
        tp, tt = evaluate(model, Xte, yte, cfg.BATCH)
        t_f1 = macro_f1(tp, tt, C); t_acc = accuracy(tp, tt)
        improved = v_smooth > best_val + 1e-4
        if improved:
            best_val, best_epoch, pc = v_smooth, ep, 0
            best = {"f1": t_f1, "acc": t_acc, "firing": firing_mean(model, fire_batch),
                    "preds": tp.tolist(), "trues": tt.tolist()}
        else:
            pc += 1
        torch.save({"epoch": ep, "model": copy.deepcopy(model.state_dict()), "opt": opt.state_dict(),
                    "m": {"val_hist": val_hist, "best_val": best_val, "best_epoch": best_epoch,
                          "pc": pc, "best": best}}, ckpt)
        cur = best["firing"] if best else 0.0
        if use_bar:
            it.set_postfix(val=f"{v_smooth:.3f}", f1=f"{t_f1:.3f}", fire=f"{cur:.2f}", pat=pc)
        elif (ep % cfg.HEARTBEAT_EVERY == 0) or improved:
            print(f"      ep{ep:>3}/{cfg.MAX_EPOCHS} val={v_smooth:.4f} f1={t_f1:.4f} "
                  f"fire={cur:.3f} pat={pc}{'  *' if improved else ''}", flush=True)
        if pc >= cfg.PATIENCE: break
    if use_bar: it.close()
    best["params"] = n_params; best["best_epoch"] = best_epoch
    return best

# ══════════════════════════════════════════════════════════════════════════════
# Run + aggregate + 2×2 closure + bootstrap
# ══════════════════════════════════════════════════════════════════════════════
RES_PATH = RUN_DIR / "results.json"
res = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
print("\n" + "═"*78 + "\n DIAT per-neuron bake-off (homogeneous, matched to ablation)\n" + "═"*78, flush=True)
for neuron in cfg.NEURONS:
    for seed in cfg.SEEDS:
        key = f"{neuron}:{seed}"
        if key in res: continue
        print(f"\n[{neuron} seed {seed}]", flush=True)
        r = train_one(neuron, seed)
        res[key] = r; RES_PATH.write_text(json.dumps(res))
        print(f"   -> {key}: F1={r['f1']:.4f} acc={r['acc']:.4f} fire={r['firing']:.3f} "
              f"@ep{r['best_epoch']} params={r['params']}", flush=True)

def agg(neuron):
    ks = [f"{neuron}:{s}" for s in cfg.SEEDS if f"{neuron}:{s}" in res]
    return ks, [res[k]["f1"] for k in ks], [res[k]["acc"] for k in ks], [res[k]["firing"] for k in ks]

print("\n" + "═"*78 + "\n DIAT RANKING  (macro-F1, mean±std over seeds)\n" + "═"*78, flush=True)
diat_now = {}
for neuron in cfg.NEURONS:
    ks, f1, ac, fr = agg(neuron)
    if not ks: continue
    diat_now[neuron] = float(np.mean(f1))
    print(f"   {neuron:16s} F1 {np.mean(f1):.4f}±{np.std(f1):.4f}  acc {np.mean(ac):.4f}  fire {np.mean(fr):.3f}", flush=True)

# harness-consistency check on the Doppler reproduction
dop_repro = diat_now.get("Doppler-LIF")
if dop_repro is not None:
    print(f"\n[CHECK] Doppler-LIF DIAT reproduction: {dop_repro:.4f}  (locked ablation 0.8719, "
          f"Δ={dop_repro-0.8719:+.4f})", flush=True)

# ── the 2×2 ────────────────────────────────────────────────────────────────────
print("\n" + "═"*78 + "\n DOUBLE DISSOCIATION 2×2  (winner of each domain)\n" + "═"*78, flush=True)
print(f"   {'neuron':16s}   DIAT-F1    SHD-F1", flush=True)
for n in ["Doppler-LIF", "CrossInhib-LIF"]:
    d = diat_now.get(n, DIAT_HOMOG.get(n)); s = SHD_HOMOG.get(n)
    print(f"   {n:16s}   {d:.4f}    {s:.4f}", flush=True)
ciP = diat_now.get("Phase-LIF")
if ciP is not None:
    print(f"   {'Phase-LIF':16s}   {ciP:.4f}    {SHD_HOMOG['Phase-LIF']:.4f}", flush=True)

dopD, ciD = diat_now.get("Doppler-LIF"), diat_now.get("CrossInhib-LIF")
ciS, dopS = SHD_HOMOG["CrossInhib-LIF"], SHD_HOMOG["Doppler-LIF"]
square = (dopD is not None and ciD is not None and dopD > ciD and ciS > dopS)
print(f"\n   DIAT: Doppler {dopD:.4f} vs CrossInhib {ciD:.4f}  -> "
      f"{'Doppler wins' if dopD>ciD else 'CrossInhib wins'}", flush=True)
print(f"   SHD : CrossInhib {ciS:.4f} vs Doppler {dopS:.4f}  -> "
      f"{'CrossInhib wins' if ciS>dopS else 'Doppler wins'}", flush=True)
print(f"   2×2 DOUBLE DISSOCIATION {'CONFIRMED' if square else 'NOT confirmed'} "
      f"(each domain won by its physics-matched neuron)", flush=True)

print("\n[BOOTSTRAP] paired on DIAT test, per seed (consistent = same sign all seeds)", flush=True)
boot = {}
for a, b in [("Doppler-LIF", "CrossInhib-LIF"), ("Doppler-LIF", "Phase-LIF")]:
    if not (any(f"{a}:{s}" in res for s in cfg.SEEDS) and any(f"{b}:{s}" in res for s in cfg.SEEDS)): continue
    for label, mfn in [("F1", METRICS["macro_f1"]), ("acc", METRICS["accuracy"])]:
        per = []
        for s in cfg.SEEDS:
            ka, kb = f"{a}:{s}", f"{b}:{s}"
            if ka in res and kb in res:
                per.append(paired_bootstrap(np.array(res[ka]["preds"]), np.array(res[kb]["preds"]),
                                            np.array(res[ka]["trues"]), mfn, C))
        if not per: continue
        mean = float(np.mean([x[0] for x in per])); cis = [(round(l,3), round(h,3)) for _,l,h in per]
        consistent = all(l>0 for _,l,_ in per) or all(h<0 for _,_,h in per)
        boot[f"{a}-{b}[{label}]"] = {"mean": mean, "cis": cis, "consistent": consistent}
        print(f"   {a} - {b} [{label}]: mean {mean:+.4f}  CIs {cis}  consistent={consistent}", flush=True)

summary = {"run_dir": str(RUN_DIR), "diat_now": diat_now, "doppler_reproduction": dop_repro,
           "diat_homog_locked": DIAT_HOMOG, "shd_homog": SHD_HOMOG,
           "square_confirmed": bool(square), "bootstrap": boot, "timestamp": datetime.now().isoformat()}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"\n[DONE] summary -> {RUN_DIR / 'summary.json'}", flush=True)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SHD SPEAKER-DISJOINT  ·  per-neuron ranking re-run  (reviewer fix #2)
# ───────────────────────────────────────────────────────────────────────────
# WHY: the paper's audio ranking (Table 4) and the double dissociation (Table 5)
# were measured on a speaker-MIXED re-split cache. A reviewer will ask whether
# the ranking survives the OFFICIAL speaker-DISJOINT split. This cell changes
# ONLY the split (official zenkelab.org SHD train/test, disjoint by construction:
# 2 of 12 speakers appear only in test) and keeps EVERYTHING else identical to
# the ablation/dissociation harness: L=4, W=128, pure-SNN (no norm), init-only
# threshold calibration to target firing, Adam+CE, temporal-mean rate readout,
# val-smoothed-macro-F1 selection (mean of last 5 epochs) + patience early stop,
# 3 seeds, paired bootstrap (2000 resamples, "consistent" = same sign all seeds).
#
# PRE-REGISTERED PREDICTION (fixed before looking at the numbers):
#   - Absolute macro-F1 DROPS on the disjoint split (it is the harder, correct split).
#   - The neuron RANKING is preserved: CrossInhib-LIF stays at/near the top and
#     Doppler-LIF stays near the bottom on audio.
#   - Headline statistics: (a) Spearman rank correlation between the locked
#     mixed-split ranking (Table 4) and this disjoint ranking; (b) the audio-side
#     CrossInhib-LIF vs Doppler-LIF paired bootstrap, expected consistent in sign.
#   A high Spearman + a consistent CrossInhib>Doppler on audio confirms the
#   dissociation is split-invariant. A scrambled ranking would falsify it.
#
# Single paste-and-run Colab cell. Checkpoints to Drive → resumes after a drop.
# ═══════════════════════════════════════════════════════════════════════════
import os, sys, json, gzip, shutil, time, copy, subprocess, warnings
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
warnings.filterwarnings("ignore")

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False


# ═══════════════════════════════════════════════════════════════════════════
# Config (no argparse) + experiment manifest
# ═══════════════════════════════════════════════════════════════════════════
@dataclass
class Config:
    # architecture (LOCKED — identical to the ablation/dissociation harness)
    L: int = 4
    W: int = 128
    norm: str = "none"                 # PURE SNN: no normalization anywhere
    target_firing: float = 0.15        # init threshold calibration target
    stft_window: int = 8               # SHD T=12 -> STFT window must be <= T

    # SHD binning
    T: int = 12                        # frames (matches the paper's cached SHD)
    F: int = 700                       # cochlear channels
    duration_s: float = 1.0            # bin spike times over [0, duration] into T frames

    # training / selection (LOCKED)
    max_epochs: int = 200
    patience: int = 20                 # early stop on smoothed-val plateau
    val_smooth: int = 5                # mean over last 5 epochs (paper's wording)
    val_frac: float = 0.15             # val carved from official train (selection ONLY)
    select_metric: str = "macro_f1"
    lr: float = 1e-3
    batch_size: int = 128
    seeds: list = field(default_factory=lambda: [42, 123, 999])
    nboot: int = 2000

    # the 8 neurons of Table 4 (Vanilla-LIF registered below; rest are MS-IF)
    methods: list = field(default_factory=lambda: [
        "CrossInhib-LIF", "Vanilla-LIF", "Dual-tau-LIF", "Chirp-LIF",
        "Beam-IF", "Doppler-LIF", "STFT-IF", "Phase-LIF"])

    # locked speaker-MIXED ranking (Table 4) — reference for the Spearman check.
    # (values are the paper's reported macro-F1; used ONLY for rank correlation)
    mixed_ranking: dict = field(default_factory=lambda: {
        "CrossInhib-LIF": 0.741, "Vanilla-LIF": 0.703, "Dual-tau-LIF": 0.694,
        "Chirp-LIF": 0.662, "Beam-IF": 0.639, "Doppler-LIF": 0.601,
        "STFT-IF": 0.244, "Phase-LIF": 0.143})

    # experiment manifest (single source of truth)
    exp_index: int = 2
    exp_total: int = 3
    exp_name: str = "SHD speaker-disjoint per-neuron ranking"
    exp_manifest: list = field(default_factory=lambda: [
        (1, "DIAT ablation + dissociation (mixed-split) [paper]", "done"),
        (2, "SHD speaker-DISJOINT ranking re-run", "THIS"),
        (3, "parameter-matched CNN baseline", "pending"),
    ])

cfg = Config()

# fast sandbox check:  REDUCED=1 python shd_disjoint_ranking.py
if os.environ.get("REDUCED") == "1":
    cfg.L = 2; cfg.W = 24
    cfg.max_epochs = 3; cfg.patience = 2; cfg.val_smooth = 2
    cfg.seeds = [42]; cfg.nboot = 200
    cfg.methods = ["CrossInhib-LIF", "Doppler-LIF", "STFT-IF", "Vanilla-LIF"]


# ═══════════════════════════════════════════════════════════════════════════
# Paths
# ═══════════════════════════════════════════════════════════════════════════
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RESEARCH = Path("/content/drive/MyDrive/Research")
else:
    RESEARCH = Path("/home/claude/research")
RESEARCH.mkdir(parents=True, exist_ok=True)
PROJECT = RESEARCH / "NISAC_DeepHetero"
PROJECT.mkdir(parents=True, exist_ok=True)
CACHE = PROJECT / "shd_official"
CACHE.mkdir(parents=True, exist_ok=True)


# ═══════════════════════════════════════════════════════════════════════════
# dikmen-spiking-neurons import (mandatory sys.modules cleanup + GitHub fallback)
# ═══════════════════════════════════════════════════════════════════════════
def _clear_dikmen():
    for k in [k for k in sys.modules if k.startswith("dikmen")]:
        del sys.modules[k]

_clear_dikmen()
try:
    import dikmen_neurons  # noqa
except ImportError:
    url = "git+https://github.com/DrCanD/dikmen-spiking-neurons.git"
    print("[SETUP] installing dikmen-spiking-neurons from GitHub ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", url])
    if r.returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--break-system-packages", url], check=True)
    _clear_dikmen()

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt   # Colab handles backend; never matplotlib.use('Agg')
import dikmen_neurons as D
from dikmen_neurons import BaseNeuron, spike_hard, NeuronRegistry

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# ═══════════════════════════════════════════════════════════════════════════
# Matched Vanilla-LIF baseline  (same as the HPO/ablation: zero learnable neuron params)
# ═══════════════════════════════════════════════════════════════════════════
class VanillaLIF(BaseNeuron):
    _family = "lif"
    _description = "Standard leaky integrate-and-fire baseline (control)."
    def single_step(self, x_t, state):
        mem = self.beta * state["mem"] + x_t
        spk = spike_hard(mem, self.threshold)
        mem = mem * (1.0 - spk)
        return spk, {"mem": mem}
NeuronRegistry._all["Vanilla-LIF"] = VanillaLIF


# ═══════════════════════════════════════════════════════════════════════════
# Dataset registry
# ═══════════════════════════════════════════════════════════════════════════
REGISTRY_PATH = RESEARCH / "datasets.json"

def load_registry():
    if REGISTRY_PATH.exists():
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return {}

def register_dataset(name, path, **meta):
    reg = load_registry()
    reg[name] = {"path": str(path), "registered": datetime.now().isoformat(), **meta}
    with open(REGISTRY_PATH, "w") as f:
        json.dump(reg, f, indent=2)
    print(f"[REGISTRY] {name} -> {path}")


# ═══════════════════════════════════════════════════════════════════════════
# Fast download (aria2c > wget > requests)
# ═══════════════════════════════════════════════════════════════════════════
def fast_download(url, dest, desc="file"):
    dest = Path(dest)
    if dest.exists():
        print(f"[CACHE] {desc} already at {dest}")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not shutil.which("aria2c") and IN_COLAB:
        subprocess.run(["apt-get", "install", "-y", "aria2", "-qq"], capture_output=True)
    if shutil.which("aria2c"):
        subprocess.run(["aria2c", "-x", "16", "-s", "16", "-k", "1M",
                        "--dir", str(dest.parent), "--out", dest.name, url], check=True)
    elif shutil.which("wget"):
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(dest), url], check=True)
    else:
        import urllib.request
        urllib.request.urlretrieve(url, dest)
    print(f"[OK] {desc} -> {dest}")
    return dest


# ═══════════════════════════════════════════════════════════════════════════
# OFFICIAL SHD loader  (speaker-disjoint train/test; bin spikes -> [N, T, 700])
# ═══════════════════════════════════════════════════════════════════════════
# Mirrors: zenkelab.org/datasets and compneuro.net/datasets
SHD_URLS = {
    "train": ["https://zenkelab.org/datasets/shd_train.h5.gz",
              "https://compneuro.net/datasets/shd_train.h5.gz"],
    "test":  ["https://zenkelab.org/datasets/shd_test.h5.gz",
              "https://compneuro.net/datasets/shd_test.h5.gz"],
}

def _download_shd(split):
    gz = CACHE / f"shd_{split}.h5.gz"
    h5 = CACHE / f"shd_{split}.h5"
    if not h5.exists():
        last = None
        for u in SHD_URLS[split]:
            try:
                fast_download(u, gz, desc=f"SHD {split}")
                last = None
                break
            except Exception as e:
                last = e
                print(f"[WARN] {u} failed: {e}")
        if last is not None:
            raise last
        with gzip.open(gz, "rb") as fi, open(h5, "wb") as fo:
            shutil.copyfileobj(fi, fo)
    return h5

def _bin_h5(h5_path, T, F, duration):
    """SHD HDF5 -> [N, T, F] spike-count frames. Keys: spikes/times, spikes/units, labels."""
    import h5py
    with h5py.File(h5_path, "r") as f:
        times = f["spikes"]["times"]
        units = f["spikes"]["units"]
        labels = np.asarray(f["labels"], dtype=np.int64)
        N = len(labels)
        X = np.zeros((N, T, F), dtype=np.float32)
        for i in range(N):
            t = np.asarray(times[i], dtype=np.float32)
            u = np.asarray(units[i], dtype=np.int64)
            if t.size == 0:
                continue
            b = np.clip((t / duration * T).astype(np.int64), 0, T - 1)
            u = np.clip(u, 0, F - 1)
            np.add.at(X, (np.full(b.shape, i), b, u), 1.0)
    return X, labels

def _strat_split(X, y, frac, seed):
    try:
        from sklearn.model_selection import train_test_split
        A, B, ya, yb = train_test_split(X, y, test_size=frac, stratify=y, random_state=seed)
        return A, ya, B, yb
    except Exception:
        rng = np.random.default_rng(seed); ia, ib = [], []
        for c in np.unique(y):
            ci = np.where(y == c)[0]; rng.shuffle(ci); k = int((1 - frac) * len(ci))
            ia += list(ci[:k]); ib += list(ci[k:])
        return X[ia], y[ia], X[ib], y[ib]

def load_shd_disjoint():
    """Returns train/val/test tensors. test = OFFICIAL disjoint test; val carved from train."""
    if IN_COLAB:
        tr_h5, te_h5 = _download_shd("train"), _download_shd("test")
    else:
        # sandbox: synthetic SHD-structured HDF5 (only to verify the pipeline runs)
        tr_h5, te_h5 = _make_synth_h5("train"), _make_synth_h5("test")
    Xtr_full, ytr_full = _bin_h5(tr_h5, cfg.T, cfg.F, cfg.duration_s)
    Xte, yte = _bin_h5(te_h5, cfg.T, cfg.F, cfg.duration_s)
    # carve a stratified validation set from the official TRAIN (selection only)
    Xtr, ytr, Xva, yva = _strat_split(Xtr_full, ytr_full, cfg.val_frac, seed=0)
    register_dataset("shd_official",
        path=str(CACHE), source="zenkelab.org", format="hdf5", split="speaker-disjoint",
        classes=int(max(ytr_full.max(), yte.max()) + 1), T=cfg.T, features=cfg.F,
        notes="official train/test; 2 of 12 speakers test-only; binned to T frames")
    to_t = lambda a: torch.as_tensor(a).float()
    to_y = lambda a: torch.as_tensor(a).long()
    return (to_t(Xtr), to_y(ytr), to_t(Xva), to_y(yva), to_t(Xte), to_y(yte))

def _make_synth_h5(split):
    """Tiny synthetic file with the real SHD layout, for the local REDUCED check only."""
    import h5py
    p = CACHE / f"synth_{split}.h5"
    if p.exists():
        return p
    rng = np.random.default_rng(0 if split == "train" else 1)
    N, C = (240 if split == "train" else 80), 20
    y = rng.integers(0, C, N).astype(np.int64)
    dt_t = h5py.special_dtype(vlen=np.dtype("float32"))
    dt_u = h5py.special_dtype(vlen=np.dtype("int64"))
    with h5py.File(p, "w") as f:
        g = f.create_group("spikes")
        ts = g.create_dataset("times", (N,), dtype=dt_t)
        us = g.create_dataset("units", (N,), dtype=dt_u)
        for i in range(N):
            k = rng.integers(40, 120)
            # class-correlated channel band so a classifier can actually learn
            lo = (y[i] * 700 // C)
            ts[i] = np.sort(rng.uniform(0, 1.0, k)).astype(np.float32)
            us[i] = np.clip(rng.normal(lo, 60, k), 0, 699).astype(np.int64)
        f.create_dataset("labels", data=y)
    return p


# ═══════════════════════════════════════════════════════════════════════════
# Deep bodies (inline) — verbatim from the ablation/dissociation harness (norm="none")
# ═══════════════════════════════════════════════════════════════════════════
def _make_norm(kind, width):
    if kind == "layer": return nn.LayerNorm(width)
    if kind == "batch": return nn.BatchNorm1d(width)
    return nn.Identity()                       # PURE SNN default

class FeatureStack(nn.Module):
    def __init__(self, in_features, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        neuron_kwargs = neuron_kwargs or {}
        dims = [in_features] + [width] * len(layer_types)
        self.projs = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(layer_types))])
        self.norms = nn.ModuleList([_make_norm(norm, width) for _ in layer_types])
        self.neurons = nn.ModuleList(
            [NeuronRegistry.create(t, size=width, **neuron_kwargs.get(t, {})) for t in layer_types])
        self.width = width
    def forward(self, x):
        h = x
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); h = spk
        return h.mean(dim=1), h
    def layer_firing(self, x):
        h, rates = x, []
        for proj, norm, neuron in zip(self.projs, self.norms, self.neurons):
            B, T, d = h.shape
            z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
            spk, _ = neuron(z); rates.append(spk.float().mean().item()); h = spk
        return rates

class VerticalNet(nn.Module):
    def __init__(self, in_features, n_classes, layer_types, width, neuron_kwargs=None, norm="none"):
        super().__init__()
        self.stack = FeatureStack(in_features, layer_types, width, neuron_kwargs, norm)
        self.readout = nn.Linear(width, n_classes)
        self.layer_types = list(layer_types)
    def forward(self, x):
        feat, _ = self.stack(x); return self.readout(feat)
    def firing_rates(self, x):
        return self.stack.layer_firing(x)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def _calibrate_stack(stack, h, target, iters=30):
    """Bisection: set each layer's threshold so init firing ~= target. The spike
    equation (mem >= threshold) is unchanged; threshold is a per-layer scalar."""
    for proj, norm, neuron in zip(stack.projs, stack.norms, stack.neurons):
        B, T, d = h.shape
        z = norm(proj(h.reshape(B*T, d))).reshape(B, T, -1)
        a, b = 1e-3, 1e5
        for _ in range(iters):
            mid = (a*b)**0.5; neuron.threshold = mid
            r = neuron(z)[0].float().mean().item()
            if r > target: a = mid
            else:          b = mid
        neuron.threshold = (a*b)**0.5
        h = neuron(z)[0]
    return stack

@torch.no_grad()
def calibrate_thresholds(model, x, target):
    _calibrate_stack(model.stack, x, target)
    return model


# ═══════════════════════════════════════════════════════════════════════════
# Metrics / eval / bootstrap  (verbatim from the ablation)
# ═══════════════════════════════════════════════════════════════════════════
def accuracy(p, t, C=None): return float((p == t).mean())

def macro_f1(p, t, C):
    f1s = []
    for c in range(C):
        tp = int(np.sum((p == c) & (t == c))); fp = int(np.sum((p == c) & (t != c)))
        fn = int(np.sum((p != c) & (t == c)))
        pr = tp/(tp+fp) if (tp+fp) else 0.0; rc = tp/(tp+fn) if (tp+fn) else 0.0
        f1s.append(2*pr*rc/(pr+rc) if (pr+rc) else 0.0)
    return float(np.mean(f1s)), [round(float(x), 4) for x in f1s]

METRICS = {"accuracy": lambda p, t, C: accuracy(p, t),
           "macro_f1": lambda p, t, C: macro_f1(p, t, C)[0]}

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); preds, trues = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(device)).argmax(1).cpu()); trues.append(yb)
    return torch.cat(preds).numpy(), torch.cat(trues).numpy()

def firing_snapshot(model, xb):
    return model.firing_rates(xb)

def paired_bootstrap(preds_a, preds_b, trues, metric_fn, C, nboot=2000, seed=0):
    """Paired bootstrap over test examples; resamples preds AND trues by the same indices."""
    rng = np.random.default_rng(seed); n = len(trues)
    base = metric_fn(preds_a, trues, C) - metric_fn(preds_b, trues, C)
    boots = np.empty(nboot)
    for k in range(nboot):
        i = rng.integers(0, n, n)
        boots[k] = metric_fn(preds_a[i], trues[i], C) - metric_fn(preds_b[i], trues[i], C)
    return float(base), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


# ═══════════════════════════════════════════════════════════════════════════
# Model builder (8 homogeneous single-neuron nets) + train_one (verbatim)
# ═══════════════════════════════════════════════════════════════════════════
def build_model(method, Fin, C, NK):
    return VerticalNet(Fin, C, [method] * cfg.L, cfg.W, NK, cfg.norm)

def save_ckpt(model, opt, epoch, metrics, path):
    torch.save({"epoch": epoch, "model_state_dict": copy.deepcopy(model.state_dict()),
                "optimizer_state_dict": opt.state_dict(), "metrics": metrics}, path)

def load_ckpt(model, opt, path):
    if not path.exists():
        return 0, {}
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model_state_dict"]); opt.load_state_dict(ck["optimizer_state_dict"])
    print(f"    [RESUME] from epoch {ck['epoch']}")
    return ck["epoch"] + 1, ck["metrics"]

def train_one(method, seed, Fin, C, NK, tr_loader, va_loader, te_loader, fire_batch, ckpt_dir):
    """Early stop on SMOOTHED validation; report TEST once at best-smoothed-val epoch."""
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = build_model(method, Fin, C, NK).to(device)
    calibrate_thresholds(model, fire_batch.to(device), cfg.target_firing)   # init only
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    ckpt = ckpt_dir / f"{method}_seed{seed}.pt"
    start, m = load_ckpt(model, opt, ckpt)
    sel = METRICS[cfg.select_metric]
    fire_log     = m.get("fire_log", [])
    val_hist     = m.get("val_hist", [])
    best_val     = m.get("best_val", -1.0)
    best_epoch   = m.get("best_epoch", -1)
    patience_ctr = m.get("patience_ctr", 0)
    best = m.get("best", None)
    for ep in range(start, cfg.max_epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward(); opt.step()
        vp, vt = evaluate(model, va_loader); v_raw = sel(vp, vt, C)
        val_hist.append(v_raw)
        v_smooth = float(np.mean(val_hist[-cfg.val_smooth:]))
        tp, tt = evaluate(model, te_loader)
        t_acc = accuracy(tp, tt); t_f1, t_f1c = macro_f1(tp, tt, C)
        fr = firing_snapshot(model, fire_batch.to(device))
        improved = v_smooth > best_val + 1e-4
        if improved:
            best_val, best_epoch, patience_ctr = v_smooth, ep, 0
            best = {"acc": t_acc, "f1": t_f1, "f1_per_class": t_f1c,
                    "firing": [round(r, 4) for r in fr],
                    "preds": tp.tolist(), "trues": tt.tolist()}
        else:
            patience_ctr += 1
        fire_log.append({"epoch": ep, "val_smooth": round(v_smooth, 4), "val_raw": round(v_raw, 4),
                         "test_acc": round(t_acc, 4), "test_f1": round(t_f1, 4),
                         "firing": [round(r, 4) for r in fr]})
        save_ckpt(model, opt, ep, {"fire_log": fire_log, "val_hist": val_hist, "best_val": best_val,
                                   "best_epoch": best_epoch, "patience_ctr": patience_ctr,
                                   "best": best}, ckpt)
        star = " *" if improved else ""
        print(f"    ep {ep:>3} val={v_smooth:.4f}(raw {v_raw:.4f}) test_acc={t_acc:.4f} test_f1={t_f1:.4f}{star}")
        if patience_ctr >= cfg.patience:
            print(f"    [EARLY STOP] no smoothed-val gain {cfg.patience} ep; best @ep{best_epoch} "
                  f"-> test f1={best['f1']:.4f}")
            break
    return best, fire_log


# ═══════════════════════════════════════════════════════════════════════════
# Rank correlation (Spearman, manual: Pearson on ranks; no scipy dependency)
# ═══════════════════════════════════════════════════════════════════════════
def spearman(names, score_x, score_y):
    xs = np.array([score_x[n] for n in names], dtype=float)
    ys = np.array([score_y[n] for n in names], dtype=float)
    rx = xs.argsort().argsort().astype(float)
    ry = ys.argsort().argsort().astype(float)
    return float(np.corrcoef(rx, ry)[0, 1])


# ═══════════════════════════════════════════════════════════════════════════
# main
# ═══════════════════════════════════════════════════════════════════════════
def main():
    RUN_ID = os.environ.get("RUN_ID", "main")   # STABLE across sessions -> resumes after a Colab drop; set RUN_ID=... or delete the dir to start fresh
    RUN_DIR = PROJECT / "runs_shd_disjoint" / RUN_ID
    CKPT_DIR = RUN_DIR / "checkpoints"
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

    # ── banner + manifest ──
    print(f"[PERSIST] project: {PROJECT}")
    print(f"[HW] device={device}  gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
    print(f"[PROGRESS] Experiment {cfg.exp_index}/{cfg.exp_total}: {cfg.exp_name}")
    for idx, name, status in cfg.exp_manifest:
        mark = "->" if status == "THIS" else ("x" if status == "done" else "  ")
        print(f"   {mark} {idx}/{cfg.exp_total}: {name} [{status}]")

    # ── data (official disjoint test; val carved from official train) ──
    Xtr, ytr, Xva, yva, Xte, yte = load_shd_disjoint()
    Fin = Xtr.shape[-1]; C = int(max(ytr.max(), yva.max(), yte.max()) + 1)
    print(f"[DATA] SHD-disjoint train={tuple(Xtr.shape)} val={tuple(Xva.shape)} test={tuple(Xte.shape)}  C={C}")
    print(f"[DATA] spike-count range [{Xtr.min():.1f}, {Xtr.max():.1f}]  chance={1/C:.3f}")
    NK = {"STFT-IF": {"window_len": cfg.stft_window}}
    fire_batch = Xtr[:cfg.batch_size]
    pin = (device.type == "cuda")
    tr_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=cfg.batch_size, shuffle=True,
                           num_workers=2, pin_memory=pin, persistent_workers=pin)
    va_loader = DataLoader(TensorDataset(Xva, yva), batch_size=256, shuffle=False)
    te_loader = DataLoader(TensorDataset(Xte, yte), batch_size=256, shuffle=False)

    # ── workload ──
    n_runs = len(cfg.methods) * len(cfg.seeds)
    print(f"[WORKLOAD] {len(cfg.methods)} neurons x {len(cfg.seeds)} seeds = {n_runs} runs x <= {cfg.max_epochs} ep")

    # ── run ──
    results = {}; run_counter = 0; first_t = None
    for method in cfg.methods:
        f1s, accs, preds_seeds, trues_seeds = [], [], [], []
        for seed in cfg.seeds:
            run_counter += 1
            print(f"\n  Run {run_counter}/{n_runs}: {method} | seed={seed}")
            t0 = time.time()
            best, flog = train_one(method, seed, Fin, C, NK,
                                   tr_loader, va_loader, te_loader, fire_batch, CKPT_DIR)
            dt = time.time() - t0
            if first_t is None:
                first_t = dt; print(f"    [ETA] ~{first_t*(n_runs-1)/60:.0f} min for remaining {n_runs-1} runs")
            f1s.append(best["f1"]); accs.append(best["acc"])
            preds_seeds.append(best["preds"]); trues_seeds.append(best["trues"])
            with open(RUN_DIR / f"results_{method}_seed{seed}.json", "w") as f:
                json.dump({"method": method, "seed": seed, "f1": best["f1"], "acc": best["acc"],
                           "f1_per_class": best["f1_per_class"], "firing": best["firing"],
                           "fire_log": flog, "time_s": dt}, f, indent=2)
        results[method] = {"f1_mean": float(np.mean(f1s)), "f1_std": float(np.std(f1s)),
                           "acc_mean": float(np.mean(accs)), "f1s": f1s,
                           "preds": preds_seeds, "trues": trues_seeds}
        print(f"  => {method}: macro-F1 {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")

    # ── disjoint ranking + Spearman vs the locked mixed ranking ──
    disjoint = {m: results[m]["f1_mean"] for m in cfg.methods}
    order_dis = sorted(cfg.methods, key=lambda m: disjoint[m], reverse=True)
    order_mix = sorted(cfg.methods, key=lambda m: cfg.mixed_ranking[m], reverse=True)
    rho = spearman(cfg.methods, cfg.mixed_ranking, disjoint)

    print("\n" + "=" * 74)
    print(" SHD per-neuron ranking: locked MIXED (Table 4)  vs  OFFICIAL DISJOINT")
    print("=" * 74)
    print(f"  {'neuron':<16}{'mixed F1':>10}{'mixed rk':>9}   {'disjoint F1':>12}{'disjoint rk':>12}")
    for m in order_mix:
        print(f"  {m:<16}{cfg.mixed_ranking[m]:>10.3f}{order_mix.index(m)+1:>9}   "
              f"{disjoint[m]:>12.4f}{order_dis.index(m)+1:>12}")
    print(f"\n  Spearman rank correlation (mixed vs disjoint) = {rho:+.3f}")
    print(f"  mixed   order: {' > '.join(order_mix)}")
    print(f"  disjoint order: {' > '.join(order_dis)}")

    # ── audio-side dissociation: CrossInhib-LIF vs Doppler-LIF, paired bootstrap per seed ──
    boot = None
    a, b = "CrossInhib-LIF", "Doppler-LIF"
    if a in results and b in results:
        per_seed = []
        for si in range(len(cfg.seeds)):
            pa = np.array(results[a]["preds"][si]); pb = np.array(results[b]["preds"][si])
            tr = np.array(results[a]["trues"][si])
            per_seed.append(paired_bootstrap(pa, pb, tr, METRICS["macro_f1"], C, nboot=cfg.nboot,
                                             seed=cfg.seeds[si]))
        diffs = [d for d, _, _ in per_seed]
        consistent = all(lo > 0 for _, lo, _ in per_seed) or all(hi < 0 for _, _, hi in per_seed)
        boot = {"pair": f"{a} - {b}", "mean_diff": float(np.mean(diffs)),
                "per_seed_ci": [(round(l, 3), round(h, 3)) for _, l, h in per_seed],
                "consistent_sign": consistent}
        print(f"\n[BOOTSTRAP audio] {a} - {b}: mean macro-F1 diff {np.mean(diffs):+.4f}  "
              f"per-seed CIs {boot['per_seed_ci']}  consistent_sign={consistent}")
        print("  (paper Table 5: CrossInhib beats Doppler on audio by ~0.140, consistent)")

    # ── summary ──
    summary = {"run_id": RUN_ID, "split": "speaker-disjoint (official zenkelab.org)",
               "config": {k: v for k, v in vars(cfg).items()},
               "results": {m: {"f1_mean": results[m]["f1_mean"], "f1_std": results[m]["f1_std"],
                               "acc_mean": results[m]["acc_mean"], "f1s": results[m]["f1s"]}
                           for m in cfg.methods},
               "disjoint_ranking": order_dis, "mixed_ranking": order_mix,
               "spearman_mixed_vs_disjoint": rho, "crossinhib_vs_doppler_audio": boot,
               "timestamp": datetime.now().isoformat()}
    for nm, base in [("summary.json", RUN_DIR), ("latest_shd_disjoint.json", PROJECT)]:
        with open(base / nm, "w") as f:
            json.dump(summary, f, indent=2)

    # ── plot: mixed vs disjoint ranking ──
    try:
        ys_mix = [cfg.mixed_ranking[m] for m in order_dis]
        ys_dis = [disjoint[m] for m in order_dis]
        x = np.arange(len(order_dis))
        plt.figure(figsize=(9, 4))
        plt.plot(x, ys_mix, "o--", label="mixed split (Table 4)")
        plt.plot(x, ys_dis, "s-", label="official disjoint split")
        plt.xticks(x, order_dis, rotation=40, ha="right", fontsize=8)
        plt.ylabel("macro-F1"); plt.title(f"SHD per-neuron ranking  (Spearman rho={rho:+.2f})")
        plt.legend(); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"[plot skipped] {e}")

    print(f"\n[DONE] summary -> {RUN_DIR / 'summary.json'}")
    return summary

summary = main()